# 03 – Train F and G, mirror-TTA, ensemble selection
Trains F (rebalance 1.0, seed 4) and G (rebalance 1.25, seed 5) in parallel, re-predicts every model (B and C from notebook 02, plus F and G) with mirror test-time augmentation, then builds the ensemble by forward selection on val. Input: the output of notebook 02. Accelerator: GPU T4 x2.

**Results (val, official scorer)**

| Model | Plain | With mirror TTA |
|---|---|---|
| B | 0.6406 | 0.6576 |
| C | 0.6608 | 0.6637 |
| F | 0.6655 | 0.6684 |
| G | 0.6761 | **0.6799** |
| **Ensemble G + F + B** (TTA) | | **0.6948** |

The post-processing stage at the end of this run failed (a scorer-input bug, fixed in notebook 04), so no tuned submission was written here.

In [1]:
import os, sys, glob, subprocess, time, traceback
if not glob.glob("/tmp/data/**/annotations", recursive=True):
    subprocess.run("pip -q install gdown", shell=True, check=True)
    subprocess.run('gdown --folder "https://drive.google.com/drive/folders/16HHHfCWM_p8RIvHQkQ38PMnZJNXzgXdU" -O /tmp/dl', shell=True, check=True)
    subprocess.run("unzip -q /tmp/dl/diver3d_v2.zip -d /tmp/data", shell=True, check=True)
ROOT = glob.glob("/tmp/data/**/annotations", recursive=True)[0].rsplit("/annotations", 1)[0]
print("ROOT =", ROOT)

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1MxBRohKXmK5ewLgNe773_SwmIqvTis_G
From (redirected): https://drive.google.com/uc?id=1MxBRohKXmK5ewLgNe773_SwmIqvTis_G&confirm=t&uuid=250768f4-b61f-4468-936f-bfc25d5a4e88
To: /tmp/dl/diver3d_v2.zip
  0%|          | 6.82M/3.45G [00:00<01:01, 56.0MB/s]

Processing file 1MxBRohKXmK5ewLgNe773_SwmIqvTis_G diver3d_v2.zip
Processing file 1xYQCMGKUypaXu9aDUCKZK0rId2BjtTSJ diver3d_v2.zip.sha256


100%|██████████| 3.45G/3.45G [00:32<00:00, 105MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1xYQCMGKUypaXu9aDUCKZK0rId2BjtTSJ
To: /tmp/dl/diver3d_v2.zip.sha256
100%|██████████| 81.0/81.0 [00:00<00:00, 309kB/s]
Download completed


ROOT = /tmp/data/diver3d_v2


In [2]:
for d in ['d3','tests','tools']: os.makedirs(f'/kaggle/working/code/{d}', exist_ok=True)

In [3]:
%%writefile /kaggle/working/code/d3/__init__.py
# d3 package


Writing /kaggle/working/code/d3/__init__.py


In [4]:
%%writefile /kaggle/working/code/d3/analyze.py
"""
Error analysis on val: where do the points go? (approximate matching by BEV center
distance -- good enough for diagnosis; final numbers always come from the official scorer)
"""
import numpy as np
from .geometry import sym_angle_deg, tilt_deg


def match(preds, gt, thr=0.3, dist=1.0):
    """Greedy per frame, highest score first. Returns list of (pred, gt) box pairs + counts."""
    pairs, n_fp, n_fp_empty, n_gt = [], 0, 0, 0
    for t, g in gt.items():
        p = preds.get(t)
        n_gt += len(g["c"])
        if p is None:
            continue
        keep = np.where(p["score"] >= thr)[0]
        keep = keep[np.argsort(-p["score"][keep])]
        free = set(range(len(g["c"])))
        for k in keep:
            best, bd = None, dist
            for j in free:
                dd = np.hypot(*(p["c"][k, :2] - g["c"][j, :2]))
                if dd < bd:
                    best, bd = j, dd
            if best is None:
                n_fp += 1
                n_fp_empty += len(g["c"]) == 0
            else:
                free.discard(best)
                pairs.append(dict(pc=p["c"][k], pd=p["d"][k], pR=p["R"][k], s=p["score"][k],
                                  gc=g["c"][best], gd=g["d"][best], gR=g["R"][best]))
    return pairs, n_fp, n_fp_empty, n_gt


def report(preds, gt, thr=0.3):
    pairs, n_fp, n_fpe, n_gt = match(preds, gt, thr)
    if not pairs:
        print("no matches"); return {}
    pR = np.array([p["pR"] for p in pairs]); gR = np.array([p["gR"] for p in pairs])
    err = sym_angle_deg(pR, gR)
    flip = np.einsum("ni,ni->n", pR[:, :, 2], gR[:, :, 2]) < 0       # head axes disagree
    gt_tilt = tilt_deg(gR); pr_tilt = tilt_deg(pR)
    dc = np.array([p["pc"] - p["gc"] for p in pairs])
    ratio = np.array([p["pd"] / p["gd"] for p in pairs])
    n_empty = sum(len(g["c"]) == 0 for g in gt.values())
    print(f"score>={thr}: recall {len(pairs)/n_gt:.3f} | FP {n_fp} ({n_fpe} in empty frames, "
          f"{n_fpe/max(n_empty,1):.3f} per empty frame)")
    print(f"rotation error (sym): mean {err.mean():.1f}  median {np.median(err):.1f}")
    from .geometry import Rx
    fixed = err.copy()
    fixed[flip] = sym_angle_deg(pR[flip] @ Rx(np.pi), gR[flip])
    print(f"  head<->feet flips: {flip.mean()*100:.1f}% of matches. If every flip were fixed, mean err "
          f"{err.mean():.1f} -> {fixed.mean():.1f} deg  (= +{(err.mean()-fixed.mean())*0.00185:.4f} SDS, upper bound)")
    for lo, hi, name in ((0, 15, "upright"), (15, 75, "in-between"), (75, 105, "horizontal"), (105, 181, "head-down")):
        m = (gt_tilt >= lo) & (gt_tilt < hi)
        if m.sum():
            print(f"  GT {name:10s} n={m.sum():5d}  rot err {err[m].mean():5.1f}  flips {flip[m].mean()*100:5.1f}%  "
                  f"pred head-down {np.mean(pr_tilt[m] > 105)*100:5.1f}%")
    print(f"predicted head-down (tilt>105): {np.mean(pr_tilt>105)*100:.1f}% vs GT {np.mean(gt_tilt>105)*100:.1f}%")
    print(f"center error  bias xyz {dc.mean(0).round(3)}  std xyz {dc.std(0).round(3)}")
    print(f"dims pred/gt  median l,w,h {np.median(ratio,0).round(3)}  (>1 = we predict too big)")
    return dict(err=err, flip=flip, dc=dc, ratio=ratio)


Writing /kaggle/working/code/d3/analyze.py


In [5]:
%%writefile /kaggle/working/code/d3/config.py
"""All hyperparameters in one place. Override from the command line with
    --set key=value key2=value2     (values are parsed with Python literal syntax)
"""
import ast
from dataclasses import dataclass, asdict, fields


@dataclass
class Cfg:
    name: str = "v1"
    seed: int = 0
    # ---- grid: crop and voxel size (metres). 128 x 128 x 24 voxels.
    x_range: tuple = (0.0, 12.8)
    y_range: tuple = (-6.4, 6.4)
    z_range: tuple = (-3.0, 3.0)
    vox: tuple = (0.1, 0.1, 0.25)
    out_stride: int = 2               # BEV output cell = 0.2 m  -> 64 x 64 map
    max_points: int = 20000
    # ---- targets
    mean_dims: tuple = (0.90, 0.82, 1.76)   # train medians; we regress log(d / mean)
    min_overlap: float = 0.7                # CenterPoint gaussian radius (paper value)
    min_radius: int = 2
    max_objs: int = 8
    # ---- augmentation
    gt_sample: bool = True
    gt_target: int = 4                # fill frames up to this many divers ...
    gt_floor: int = 1                 # ... then decay by 1/epoch down to this
    gt_hold_frac: float = 0.65        # fraction of epochs at full gt_target
    gt_margin: float = 0.10           # metres around a GT box that count as "diver points"
    max_bearing_deg: float = 40.0     # sonar FOV is about +-45 deg
    paste_rot_deg: float = 30.0       # pasted diver rotated about sonar origin by up to this
    min_sep: float = 1.2              # min BEV distance between diver centers (m)
    rebalance: float = 0.0            # 0 = off, 0.5 = sqrt-tempered pose reweighting, 1 = full
    mirror_p: float = 0.5
    rot_deg: float = 15.0
    scale_lo: float = 0.97
    scale_hi: float = 1.03
    drop_max: float = 0.15
    int_jitter: float = 0.10
    # ---- model
    vfe_ch: int = 16
    bev_ch: int = 128
    # ---- loss weights
    w_hm: float = 1.0
    w_off: float = 1.0
    w_z: float = 2.0
    w_dim: float = 1.0
    w_rot: float = 1.0
    # ---- optimisation
    epochs: int = 20
    bs: int = 8
    lr: float = 2e-3
    wd: float = 1e-2
    clip: float = 10.0
    ema: float = 0.999
    workers: int = 2
    eval_every: int = 4               # + always the last 3 epochs
    # ---- decoding
    K: int = 30
    score_thr: float = 0.03
    nms_dist: float = 0.6

    # derived sizes
    @property
    def grid(self):
        nx = round((self.x_range[1] - self.x_range[0]) / self.vox[0])
        ny = round((self.y_range[1] - self.y_range[0]) / self.vox[1])
        nz = round((self.z_range[1] - self.z_range[0]) / self.vox[2])
        return nx, ny, nz

    @property
    def bev(self):
        nx, ny, _ = self.grid
        return nx // self.out_stride, ny // self.out_stride

    @property
    def cell(self):
        return self.vox[0] * self.out_stride

    def update(self, pairs):
        names = {f.name for f in fields(self)}
        for kv in pairs or []:
            k, v = kv.split("=", 1)
            assert k in names, f"unknown config key {k}"
            try:
                v = ast.literal_eval(v)
            except (ValueError, SyntaxError):
                pass
            setattr(self, k, v)
        return self

    def to_dict(self):
        return asdict(self)

    @staticmethod
    def from_dict(d):
        c = Cfg()
        for k, v in d.items():
            if hasattr(c, k):
                setattr(c, k, tuple(v) if isinstance(v, list) else v)
        return c


Writing /kaggle/working/code/d3/config.py


In [6]:
%%writefile /kaggle/working/code/d3/data.py
"""
Dataset: load -> (GT paste) -> global aug -> crop -> targets.

Randomness is seeded by (seed, epoch, index), so a run is reproducible no
matter how many DataLoader workers are used, and resuming at epoch e replays
exactly the same augmentations.
"""
import math
import os
import numpy as np
import torch
from torch.utils.data import Dataset

from .geometry import (box_corners, points_in_box, tilt_deg, R_to_rot6d_zx, Rz)
from . import transforms as T
from .io import load_bin, frame_path

# ----------------------------------------------------------------------------- pose weights
# Tilt bins: [0,15) upright, [15,75) in-between, [75,105] horizontal, >105 head-down.
TILT_EDGES = np.array([15.0, 75.0, 105.0])
TRAIN_FRAC = np.array([0.196, 0.282, 0.408, 0.115])   # measured on train (data_report Q1)
TARGET_FRAC = np.array([0.521, 0.315, 0.152, 0.012])  # measured on val (~= implied test mix)


def pose_weight(tilt, power):
    """Importance weight per diver. power=0 -> all 1; power=1 -> match val mix exactly;
    power=0.5 -> geometric middle (keeps rare poses alive)."""
    b = np.searchsorted(TILT_EDGES, np.asarray(tilt), side="right")
    return (TARGET_FRAC[b] / TRAIN_FRAC[b]) ** power


# ----------------------------------------------------------------------------- GT database
class GTDatabase:
    """Points of every train diver (box enlarged by `margin`), stored as one big array."""

    def __init__(self, root, gt, cache, margin=0.10, min_pts=30):
        if os.path.exists(cache):
            z = np.load(cache)
            self.__dict__.update({k: z[k] for k in z.files})
            return
        pts_all, off, C, D, Rs = [], [0], [], [], []
        for tok, v in gt.items():
            if len(v["c"]) == 0:
                continue
            p = load_bin(frame_path(root, tok))
            for c, d, R in zip(v["c"], v["d"], v["R"]):
                m = points_in_box(p, c, d, R, margin)
                if m.sum() < min_pts:
                    continue
                pts_all.append(p[m]); off.append(off[-1] + int(m.sum()))
                C.append(c); D.append(d); Rs.append(R)
        self.pts = np.concatenate(pts_all).astype(np.float32)
        self.off = np.array(off, np.int64)
        self.c, self.d, self.R = np.array(C), np.array(D), np.array(Rs)
        self.tilt = tilt_deg(self.R)
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        tmp = cache + ".tmp.npz"                       # atomic: never leave a half-written cache
        np.savez(tmp, pts=self.pts, off=self.off, c=self.c, d=self.d, R=self.R, tilt=self.tilt)
        os.replace(tmp, cache)

    def __len__(self):
        return len(self.c)

    def points(self, i):
        return self.pts[self.off[i]:self.off[i + 1]]


# ----------------------------------------------------------------------------- heatmap helpers
def gaussian_radius(h, w, min_overlap):
    """CenterPoint/CornerNet radius (in cells) so a box shifted by r keeps IoU >= min_overlap."""
    a1, b1, c1 = 1, h + w, w * h * (1 - min_overlap) / (1 + min_overlap)
    r1 = (b1 + math.sqrt(b1 ** 2 - 4 * a1 * c1)) / 2
    a2, b2, c2 = 4, 2 * (h + w), (1 - min_overlap) * w * h
    r2 = (b2 + math.sqrt(b2 ** 2 - 4 * a2 * c2)) / 2
    a3, b3, c3 = 4 * min_overlap, -2 * min_overlap * (h + w), (min_overlap - 1) * w * h
    r3 = (b3 + math.sqrt(b3 ** 2 - 4 * a3 * c3)) / 2
    return min(r1, r2, r3)


def draw_gaussian(hm, cx, cy, r):
    d = 2 * r + 1
    sigma = d / 6
    ys, xs = np.ogrid[-r:r + 1, -r:r + 1]
    g = np.exp(-(xs * xs + ys * ys) / (2 * sigma * sigma))   # g[dx, dy] indexing via ogrid
    H, W = hm.shape
    l, rr = min(cx, r), min(H - cx, r + 1)
    t, b = min(cy, r), min(W - cy, r + 1)
    np.maximum(hm[cx - l:cx + rr, cy - t:cy + b], g[r - l:r + rr, r - t:r + b],
               out=hm[cx - l:cx + rr, cy - t:cy + b])


def build_targets(c, d, R, cfg):
    """Heatmap (1, NXc, NYc) indexed [ix, iy]; per-object ind/mask/reg.
    reg = [off_x, off_y, z, log(l/l0), log(w/w0), log(h/h0), ez(3), ex(3)]  (12)"""
    NXc, NYc = cfg.bev
    cell, x0, y0 = cfg.cell, cfg.x_range[0], cfg.y_range[0]
    hm = np.zeros((1, NXc, NYc), np.float32)
    ind = np.zeros(cfg.max_objs, np.int64)
    msk = np.zeros(cfg.max_objs, np.float32)
    reg = np.zeros((cfg.max_objs, 12), np.float32)
    k = 0
    for ci, di, Ri in zip(c, d, R):
        fx, fy = (ci[0] - x0) / cell, (ci[1] - y0) / cell
        ix, iy = int(math.floor(fx)), int(math.floor(fy))
        if not (0 <= ix < NXc and 0 <= iy < NYc) or k >= cfg.max_objs:
            continue
        cor = box_corners(ci, di, Ri)
        r = max(cfg.min_radius, int(gaussian_radius(np.ptp(cor[:, 0]) / cell, np.ptp(cor[:, 1]) / cell,
                                                    cfg.min_overlap)))
        draw_gaussian(hm[0], ix, iy, r)
        ind[k] = ix * NYc + iy
        msk[k] = 1
        reg[k] = np.concatenate([[fx - ix, fy - iy, ci[2]], np.log(di / np.array(cfg.mean_dims)),
                                 R_to_rot6d_zx(Ri)])
        k += 1
    return hm, ind, msk, reg


# ----------------------------------------------------------------------------- dataset
class DiverDataset(Dataset):
    def __init__(self, root, gt, cfg, tokens=None, train=False, db=None):
        self.root, self.gt, self.cfg, self.train, self.db = root, gt, cfg, train, db
        self.tokens = list(tokens if tokens is not None else gt.keys())
        self.epoch = 0
        if db is not None:
            w = pose_weight(db.tilt, cfg.rebalance)
            self.db_p = w / w.sum()

    def set_epoch(self, e):
        self.epoch = e

    def frame_weights(self):
        """For WeightedRandomSampler: mean pose weight of the frame's divers (empty = 1)."""
        out = []
        for t in self.tokens:
            v = self.gt[t]
            out.append(1.0 if len(v["c"]) == 0 else float(np.mean(pose_weight(tilt_deg(v["R"]), self.cfg.rebalance))))
        return np.array(out)

    def gt_target_now(self):
        cfg = self.cfg
        hold = int(round(cfg.gt_hold_frac * cfg.epochs))
        if self.epoch < hold:
            return cfg.gt_target
        return max(cfg.gt_floor, cfg.gt_target - (self.epoch - hold + 1))

    def __len__(self):
        return len(self.tokens)

    # ---- GT paste
    def _paste(self, pts, c, d, R, n, rng):
        cfg = self.cfg
        pts_l, C, D, RR = [pts], list(c), list(d), list(R)
        maxb = math.radians(cfg.max_bearing_deg)
        for _ in range(n):
            for _try in range(15):
                i = rng.choice(len(self.db), p=self.db_p)
                ci, di, Ri = self.db.c[i], self.db.d[i], self.db.R[i]
                phi = rng.uniform(-1, 1) * math.radians(cfg.paste_rot_deg)
                if abs(math.atan2(ci[1], ci[0]) + phi) > maxb:
                    continue
                Tm = Rz(phi)
                cn, Rn = Tm @ ci, Tm @ Ri
                if not (cfg.x_range[0] + 0.5 < cn[0] < cfg.x_range[1] - 0.5 and
                        cfg.y_range[0] + 0.5 < cn[1] < cfg.y_range[1] - 0.5 and
                        cfg.z_range[0] < cn[2] < cfg.z_range[1]):
                    continue
                if any(np.hypot(*(cn[:2] - cj[:2])) < cfg.min_sep for cj in C):
                    continue
                p = self.db.points(i).copy()
                p[:, :3] = p[:, :3] @ Tm.T
                # remove scene points that would sit inside the pasted diver
                pts_l = [q[~points_in_box(q, cn, di, Rn, cfg.gt_margin)] for q in pts_l]
                pts_l.append(p); C.append(cn); D.append(di); RR.append(Rn)
                break
        return np.concatenate(pts_l), np.array(C).reshape(-1, 3), np.array(D).reshape(-1, 3), \
            np.array(RR).reshape(-1, 3, 3)

    def __getitem__(self, i):
        cfg = self.cfg
        tok = self.tokens[i]
        rng = np.random.default_rng([cfg.seed, self.epoch if self.train else 0, i])
        pts = load_bin(frame_path(self.root, tok))
        v = self.gt.get(tok)
        if v is None:
            c, d, R = np.zeros((0, 3)), np.zeros((0, 3)), np.zeros((0, 3, 3))
        else:
            c, d, R = v["c"].copy(), v["d"].copy(), v["R"].copy()

        if self.train:
            if cfg.gt_sample and self.db is not None:
                n = self.gt_target_now() - len(c)
                if n > 0:
                    pts, c, d, R = self._paste(pts, c, d, R, n, rng)
            if rng.random() < cfg.mirror_p:
                pts, c, R = T.mirror_y(pts, c, R)
            if cfg.rot_deg > 0:
                pts, c, R = T.rotate_z(pts, c, R, math.radians(rng.uniform(-cfg.rot_deg, cfg.rot_deg)))
            s = rng.uniform(cfg.scale_lo, cfg.scale_hi)
            pts, c, d = T.scale(pts, c, d, s)
            if cfg.drop_max > 0:
                pts = pts[rng.random(len(pts)) >= rng.uniform(0, cfg.drop_max)]
            if cfg.int_jitter > 0:
                pts = pts.copy()
                pts[:, 3] = np.clip(pts[:, 3] * rng.uniform(1 - cfg.int_jitter, 1 + cfg.int_jitter), 0, 1)

        m = ((pts[:, 0] >= cfg.x_range[0]) & (pts[:, 0] < cfg.x_range[1]) &
             (pts[:, 1] >= cfg.y_range[0]) & (pts[:, 1] < cfg.y_range[1]) &
             (pts[:, 2] >= cfg.z_range[0]) & (pts[:, 2] < cfg.z_range[1]))
        pts = pts[m]
        if len(pts) > cfg.max_points:
            pts = pts[rng.choice(len(pts), cfg.max_points, replace=False)]
        hm, ind, msk, reg = build_targets(c, d, R, cfg)
        return dict(token=tok, pts=torch.from_numpy(np.ascontiguousarray(pts, np.float32)),
                    hm=torch.from_numpy(hm), ind=torch.from_numpy(ind),
                    msk=torch.from_numpy(msk), reg=torch.from_numpy(reg))


def collate(batch):
    n = max(1, max(len(b["pts"]) for b in batch))
    pts = torch.zeros(len(batch), n, 4)
    pmsk = torch.zeros(len(batch), n, dtype=torch.bool)
    for k, b in enumerate(batch):
        pts[k, :len(b["pts"])] = b["pts"]
        pmsk[k, :len(b["pts"])] = True
    out = dict(token=[b["token"] for b in batch], pts=pts, pmsk=pmsk)
    for key in ("hm", "ind", "msk", "reg"):
        out[key] = torch.stack([b[key] for b in batch])
    return out


Writing /kaggle/working/code/d3/data.py


In [7]:
%%writefile /kaggle/working/code/d3/decode.py
"""
Turn head maps into boxes: heatmap peaks -> read the other heads at that cell.
Then a simple distance NMS (divers are well separated; min GT separation is ~1 m).
"""
import numpy as np
import torch
import torch.nn.functional as F
from .rot_torch import rot6d_zx_to_R


@torch.no_grad()
def decode(out, cfg, K=None, thr=None):
    K = K or cfg.K
    thr = cfg.score_thr if thr is None else thr
    hm = out["hm"].float().sigmoid()[:, 0]                      # (B,H,W)
    B, H, W = hm.shape
    peak = hm * (hm == F.max_pool2d(hm[:, None], 3, 1, 1)[:, 0]).float()
    sc, idx = peak.view(B, -1).topk(min(K, H * W))
    ix, iy = idx // W, idx % W
    def g(name):
        f = out[name].float()
        return f.view(B, f.shape[1], -1).gather(2, idx[:, None].expand(-1, f.shape[1], -1)).permute(0, 2, 1)
    off, z, dim, r6 = g("off"), g("z"), g("dim"), g("rot")
    cell, x0, y0 = cfg.cell, cfg.x_range[0], cfg.y_range[0]
    cx = x0 + (ix.float() + off[..., 0]) * cell
    cy = y0 + (iy.float() + off[..., 1]) * cell
    c = torch.stack([cx, cy, z[..., 0]], -1)
    d = torch.exp(dim) * dim.new_tensor(cfg.mean_dims)
    R = rot6d_zx_to_R(r6.reshape(-1, 6)).view(B, -1, 3, 3)
    c, d, R, sc = c.cpu().numpy(), d.cpu().numpy(), R.cpu().numpy(), sc.cpu().numpy()
    res = []
    for b in range(B):
        k = sc[b] >= thr
        res.append(dist_nms(dict(c=c[b][k], d=d[b][k], R=R[b][k], score=sc[b][k]), cfg.nms_dist))
    return res


def dist_nms(p, dist):
    """Greedy: keep highest score, drop others whose BEV center is within `dist` metres."""
    if len(p["score"]) == 0 or dist <= 0:
        return p
    order = np.argsort(-p["score"])
    keep = []
    for i in order:
        if all(np.hypot(*(p["c"][i, :2] - p["c"][j, :2])) >= dist for j in keep):
            keep.append(i)
    keep = np.array(keep, int)
    return {k: v[keep] for k, v in p.items()}


Writing /kaggle/working/code/d3/decode.py


In [8]:
%%writefile /kaggle/working/code/d3/evaluate.py
"""Thin wrapper around the OFFICIAL scorer so every number we quote is the server's."""
import os
import re
import subprocess
import sys


def run_scorer(root, pred_path, gt_path=None, verbose=False):
    gt_path = gt_path or os.path.join(root, "annotations", "gt_val.json")
    out = subprocess.run([sys.executable, os.path.join(root, "tools", "diver3d_eval.py"),
                          "--gt", gt_path, "--pred", pred_path],
                         capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr[-3000:])
    if verbose:
        print(out.stdout[-1200:])
    m = dict(re.findall(r"^\s+([A-Za-z0-9_@.]+)\s+(-?[\d.]+(?:e-?\d+)?)\s*$", out.stdout, re.M))
    return {k: float(v) for k, v in m.items()}


def fmt(m):
    keys = ["SDS", "mAP3D", "AP3D@0.50", "APBEV@0.35", "ATE_m", "ASE", "AOE3D_deg", "TILT_deg"]
    return "  ".join(f"{k} {m[k]:.4f}" for k in keys if k in m)


Writing /kaggle/working/code/d3/evaluate.py


In [9]:
%%writefile /kaggle/working/code/d3/geometry.py
"""
Diver3D geometry conventions -- the single source of truth.

Every other module imports from here. If a convention is wrong, it is wrong
in exactly one place, and tests/test_geometry.py will catch it.

CONVENTIONS (from FORMAT.md / the challenge brief)
- Sonar frame: right-handed, metres. x forward, y left, z up.
- Box = (center c, dims d=[l,w,h], rotation R).
- R = [ex ey ez] : the body axes are the COLUMNS of R, written in sonar coords.
    ex = chest normal         (extent l)
    ey = right -> left shoulder (extent w)
    ez = feet -> head         (extent h),  ez = ex x ey
  So a point p in the sonar frame has body coords  R^T (p - c),
  and a body-frame point b maps to sonar coords   c + R b.
- Quaternions are Hamilton (w, x, y, z).
- Metric symmetry: only S = Rz(pi) acting in the BODY frame is free,
  i.e. R and R @ diag(-1,-1,1) = [-ex, -ey, ez] are scored identically.
  Head<->feet (R @ Rx(pi) = [ex, -ey, -ez]) is a full 180 deg error.
"""
import numpy as np

SYM = np.diag([-1.0, -1.0, 1.0])                              # Rz(pi), body frame
MIRROR_Y = np.diag([1.0, -1.0, 1.0])                          # y -> -y reflection


# --------------------------------------------------------------------------- quaternions
def quat_to_R(q):
    """(..., 4) Hamilton (w,x,y,z) -> (..., 3, 3). Same formula as FORMAT.md."""
    q = np.asarray(q, dtype=np.float64)
    q = q / np.linalg.norm(q, axis=-1, keepdims=True)
    w, x, y, z = np.moveaxis(q, -1, 0)
    R = np.stack([
        1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y),
        2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x),
        2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y),
    ], axis=-1)
    return R.reshape(q.shape[:-1] + (3, 3))


def R_to_quat(R):
    """(..., 3, 3) -> (..., 4) Hamilton (w,x,y,z), canonicalised to w >= 0.
    Uses scipy (which returns x,y,z,w -- we reorder explicitly)."""
    from scipy.spatial.transform import Rotation
    R = np.asarray(R, dtype=np.float64)
    shp = R.shape[:-2]
    xyzw = Rotation.from_matrix(R.reshape(-1, 3, 3)).as_quat()
    q = xyzw[:, [3, 0, 1, 2]]
    q = q * np.where(q[:, :1] < 0, -1.0, 1.0)  # q and -q are the same rotation
    return q.reshape(shp + (4,))


# --------------------------------------------------------------------------- SO(3) helpers
def project_to_SO3(M):
    """Nearest rotation (Frobenius) to any 3x3 matrix, via SVD. Used for averaging."""
    U, _, Vt = np.linalg.svd(M)
    D = np.ones(M.shape[:-1])
    D[..., -1] = np.sign(np.linalg.det(U @ Vt))
    return (U * D[..., None, :]) @ Vt


def rot6d_zx_to_R(a, b):
    """Our INTERNAL 6D rotation: a ~ ez (head axis), b ~ ex (chest), Gram-Schmidt.
    Gram-Schmidt keeps the FIRST vector's direction exactly and only bends the
    second, so we put first the axis that matters most (ez: tilt + head-up).
    Returns R = [ex, ey, ez]."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ez = a / np.linalg.norm(a, axis=-1, keepdims=True)
    ex = b - np.sum(ez * b, axis=-1, keepdims=True) * ez
    ex = ex / np.linalg.norm(ex, axis=-1, keepdims=True)
    ey = np.cross(ez, ex)                # z x x = y  (right-handed, cyclic)
    return np.stack([ex, ey, ez], axis=-1)


def R_to_rot6d_zx(R):
    """Inverse of rot6d_zx_to_R: returns concat(ez, ex), shape (..., 6)."""
    R = np.asarray(R, float)
    return np.concatenate([R[..., :, 2], R[..., :, 0]], axis=-1)


def rot_angle_deg(R1, R2):
    """Plain geodesic angle between rotations, degrees."""
    tr = np.trace(np.swapaxes(R1, -1, -2) @ R2, axis1=-2, axis2=-1)
    return np.degrees(np.arccos(np.clip((tr - 1) / 2, -1, 1)))


def sym_angle_deg(R_pred, R_gt):
    """AOE3D as the metric defines it: min over S in {I, Rz(pi)} of angle(Rp^T Rg S)."""
    return np.minimum(rot_angle_deg(R_pred, R_gt), rot_angle_deg(R_pred, R_gt @ SYM))


def align_to_reference(R, R_ref):
    """Return R or R@SYM, whichever is closer to R_ref. Needed before averaging
    rotations (ensemble fusion, temporal smoothing): otherwise averaging R and its
    symmetric twin gives garbage."""
    flip = rot_angle_deg(R_ref, R @ SYM) < rot_angle_deg(R_ref, R)
    return np.where(np.asarray(flip)[..., None, None], R @ SYM, R)


def tilt_deg(R):
    """Angle between body up-axis ez and sonar z. 0 = upright, 90 = horizontal,
    180 = head down."""
    R = np.asarray(R, float)
    return np.degrees(np.arccos(np.clip(R[..., 2, 2], -1, 1)))


def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])


def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


# --------------------------------------------------------------------------- boxes
_SIGNS = np.array([[sx, sy, sz] for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)], float)


def box_corners(c, d, R):
    """8 corners in sonar frame: c + R @ (signs * d/2)."""
    return (_SIGNS * (np.asarray(d) / 2)) @ np.asarray(R).T + np.asarray(c)


def points_in_box(pts, c, d, R, margin=0.0):
    """Boolean mask of points (N,>=3) inside the oriented box (optionally enlarged)."""
    local = (np.asarray(pts)[:, :3] - np.asarray(c)) @ np.asarray(R)  # = R^T (p - c), row form
    return np.all(np.abs(local) <= np.asarray(d) / 2 + margin, axis=1)


Writing /kaggle/working/code/d3/geometry.py


In [10]:
%%writefile /kaggle/working/code/d3/infer.py
"""
Inference over a list of tokens, with optional mirror test-time augmentation.
Raw detections (low threshold, after NMS) are saved so post-processing and
ensembling can be tuned later on CPU without touching the GPU again.
"""
import gzip
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import DiverDataset, collate
from .decode import decode
from .transforms import mirror_boxes_only


@torch.no_grad()
def run_inference(model, root, tokens, cfg, device="cuda", mirror=False, bs=16, workers=2, thr=None):
    model.eval()
    ds = DiverDataset(root, {}, cfg, tokens=tokens, train=False)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=workers, collate_fn=collate)
    preds = {}
    for b in dl:
        pts, pmsk = b["pts"].to(device), b["pmsk"].to(device)
        with torch.autocast(device_type=device if device != "cpu" else "cpu", dtype=torch.float16,
                            enabled=device != "cpu"):
            out = model(pts, pmsk)
        res = decode(out, cfg, thr=thr)
        if mirror:
            pm = pts.clone(); pm[..., 1] = -pm[..., 1]
            with torch.autocast(device_type=device if device != "cpu" else "cpu", dtype=torch.float16,
                                enabled=device != "cpu"):
                outm = model(pm, pmsk)
            resm = decode(outm, cfg, thr=thr)
            for r in resm:                       # bring mirrored predictions back
                if len(r["c"]):
                    r["c"], r["R"] = mirror_boxes_only(r["c"], r["R"])
            res = [dict(a=a, m=m) for a, m in zip(res, resm)]   # fused later (ensemble step)
        for tok, r in zip(b["token"], res):
            preds[tok] = r
    return preds


def save_preds(preds, path):
    with gzip.open(path, "wb") as f:
        pickle.dump(preds, f, protocol=4)


def load_preds(path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)


Writing /kaggle/working/code/d3/infer.py


In [11]:
%%writefile /kaggle/working/code/d3/io.py
"""
Data I/O: point clouds, GT, and submission writing.

GT / predictions are held in memory as  {token: dict(c=(K,3), d=(K,3), R=(K,3,3),
score=(K,), [iid=(K,)])}  -- arrays, not lists of dicts.
"""
import gzip
import json
import os
import numpy as np
from .geometry import quat_to_R, R_to_quat


def frame_path(root, token):
    scene, frame = token.split("/")
    return os.path.join(root, "scenes", scene, "sonar", f"frame_{int(frame):06d}.bin")


def load_bin(path):
    raw = np.fromfile(path, dtype=np.float32)
    assert raw.size % 4 == 0, path
    return raw.reshape(-1, 4)


def _empty():
    return dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)), R=np.zeros((0, 3, 3)),
                score=np.zeros(0), iid=np.zeros(0, int))


def load_gt(path):
    """Returns {token: arrays} in the file's order (= acquisition order)."""
    res = json.load(open(path))["results"]
    out = {}
    for tok, boxes in res.items():
        if not boxes:
            out[tok] = _empty(); continue
        out[tok] = dict(
            c=np.array([b["center"] for b in boxes], float),
            d=np.array([b["dimensions"] for b in boxes], float),
            R=quat_to_R(np.array([b["quaternion"] for b in boxes], float)),
            score=np.ones(len(boxes)),
            iid=np.array([b.get("instance_id", -1) for b in boxes], int))
    return out


def sort_tokens(tokens):
    """Acquisition order: by scene, then by frame number."""
    return sorted(tokens, key=lambda t: (t.split("/")[0], int(t.split("/")[1])))


def write_submission(preds, path, tokens=None, max_det=100, ndigits=5):
    """preds: {token: arrays}. Writes .json (dict) or .jsonl.gz (if path ends .gz).
    Every token in `tokens` is written (empty list if no prediction)."""
    tokens = tokens if tokens is not None else list(preds.keys())
    r = lambda a: [round(float(v), ndigits) for v in a]

    def boxes_of(tok):
        p = preds.get(tok)
        if p is None or len(p["c"]) == 0:
            return []
        order = np.argsort(-p["score"])[:max_det]
        q = R_to_quat(p["R"][order])
        d = np.maximum(p["d"][order], 1e-3)          # dims must be strictly positive
        s = np.clip(p["score"][order], 0.0, 1.0)
        return [dict(center=r(p["c"][i]), dimensions=r(d[k]), quaternion=r(q[k]),
                     score=round(float(s[k]), 6)) for k, i in enumerate(order)]

    if path.endswith(".gz"):
        with gzip.open(path, "wt") as f:
            for tok in tokens:
                f.write(json.dumps({"sample_token": tok, "boxes": boxes_of(tok)}) + "\n")
    else:
        json.dump({"results": {tok: boxes_of(tok) for tok in tokens}}, open(path, "w"))
    return path


Writing /kaggle/working/code/d3/io.py


In [12]:
%%writefile /kaggle/working/code/d3/losses.py
"""
Losses. All box losses are computed only at the GT center cells (gathered by `ind`).

  heatmap : CenterNet Gaussian focal loss (alpha=2, beta=4)
  off, z, dim : L1  (z weighted x2 by default: vertical is the sonar's weak axis)
  rot     : symmetric chordal loss, min over {I, Rz(pi)} -- matches AOE3D
"""
import torch
import torch.nn.functional as F
from .rot_torch import rot6d_zx_to_R, sym_chordal_loss, sym_angle_deg


def focal_loss(logits, gt):
    p = logits.float().sigmoid().clamp(1e-4, 1 - 1e-4)
    pos = gt.eq(1).float()
    pos_l = -(torch.log(p) * (1 - p) ** 2 * pos).sum()
    neg_l = -(torch.log(1 - p) * p ** 2 * (1 - gt) ** 4 * (1 - pos)).sum()
    return (pos_l + neg_l) / pos.sum().clamp(min=1)


def gather(feat, ind):
    """feat (B,C,H,W), ind (B,M) flat index ix*W+iy  ->  (B,M,C)"""
    B, C, H, W = feat.shape
    f = feat.view(B, C, H * W).permute(0, 2, 1)
    return f.gather(1, ind[..., None].expand(-1, -1, C))


def compute_loss(out, batch, cfg):
    l_hm = focal_loss(out["hm"], batch["hm"])
    m = batch["msk"] > 0
    n = m.sum().clamp(min=1).float()
    ind, reg = batch["ind"], batch["reg"].float()
    if m.any():
        off = gather(out["off"], ind).float()[m]
        z = gather(out["z"], ind).float()[m]
        dim = gather(out["dim"], ind).float()[m]
        r6 = gather(out["rot"], ind).float()[m]
        t = reg[m]
        l_off = F.l1_loss(off, t[:, 0:2], reduction="sum") / n
        l_z = F.l1_loss(z, t[:, 2:3], reduction="sum") / n
        l_dim = F.l1_loss(dim, t[:, 3:6], reduction="sum") / n
        Rp, Rg = rot6d_zx_to_R(r6), rot6d_zx_to_R(t[:, 6:12])
        l_rot = sym_chordal_loss(Rp, Rg).sum() / n
        with torch.no_grad():
            rot_deg = sym_angle_deg(Rp, Rg).mean().item()
    else:
        zero = out["hm"].sum() * 0
        l_off = l_z = l_dim = l_rot = zero
        rot_deg = 0.0
    total = (cfg.w_hm * l_hm + cfg.w_off * l_off + cfg.w_z * l_z + cfg.w_dim * l_dim + cfg.w_rot * l_rot)
    logs = dict(hm=l_hm.item(), off=l_off.item(), z=l_z.item(), dim=l_dim.item(), rot=l_rot.item(), rot_deg=rot_deg)
    return total, logs


Writing /kaggle/working/code/d3/losses.py


In [13]:
%%writefile /kaggle/working/code/d3/model.py
"""
VoxelCenterNet: points -> 3D voxels -> dense 3D conv (z folded into channels)
-> 2D BEV encoder/decoder -> CenterPoint heads.

Tensor layouts
  dense 3D grid : (B, C, NZ, NX, NY)     conv3d sees D=z, H=x, W=y
  BEV map       : (B, C, NXc, NYc)       indexed [ix, iy]   (same as targets)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

HEADS = {"hm": 1, "off": 2, "z": 1, "dim": 3, "rot": 6}


class VFE(nn.Module):
    """Per-point MLP, then max-pool the points of each voxel (plus a log-count channel)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.NX, self.NY, self.NZ = cfg.grid
        self.register_buffer("lo", torch.tensor([cfg.x_range[0], cfg.y_range[0], cfg.z_range[0]]), persistent=False)
        self.register_buffer("vs", torch.tensor(cfg.vox, dtype=torch.float32), persistent=False)
        C = cfg.vfe_ch
        self.mlp = nn.Sequential(nn.Linear(8, 32), nn.ReLU(inplace=True), nn.Linear(32, C), nn.ReLU(inplace=True))
        self.out_ch = C + 1

    def forward(self, pts, pmsk):
        B, N, _ = pts.shape
        xyz = pts[..., :3]
        idx = torch.floor((xyz - self.lo) / self.vs).long()
        valid = pmsk & (idx[..., 0] >= 0) & (idx[..., 0] < self.NX) & (idx[..., 1] >= 0) & \
            (idx[..., 1] < self.NY) & (idx[..., 2] >= 0) & (idx[..., 2] < self.NZ)
        off = (xyz - (self.lo + (idx.float() + 0.5) * self.vs)) / self.vs     # in [-0.5, 0.5]
        rng = xyz.norm(dim=-1, keepdim=True) / 10.0
        f = self.mlp(torch.cat([xyz / 5.0, pts[..., 3:4], off, rng], -1))     # (B,N,C)
        V = self.NX * self.NY * self.NZ
        flat = idx[..., 2] * (self.NX * self.NY) + idx[..., 0] * self.NY + idx[..., 1]
        flat = torch.where(valid, flat, torch.full_like(flat, V))            # invalid -> dummy slot
        C = f.shape[-1]
        out = f.new_zeros(B, V + 1, C)
        out.scatter_reduce_(1, flat[..., None].expand(-1, -1, C), f, reduce="amax", include_self=False)
        cnt = f.new_zeros(B, V + 1).scatter_add_(1, flat, valid.float())
        out = torch.cat([out, torch.log1p(cnt)[..., None]], -1)[:, :V]
        return out.view(B, self.NZ, self.NX, self.NY, C + 1).permute(0, 4, 1, 2, 3).contiguous()


def c3(i, o, s=1):
    return nn.Sequential(nn.Conv3d(i, o, 3, s, 1, bias=False), nn.BatchNorm3d(o), nn.ReLU(inplace=True))


def c2(i, o, s=1):
    return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


def up2(i, o, s):
    return nn.Sequential(nn.ConvTranspose2d(i, o, s, s, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class VoxelCenterNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.vfe = VFE(cfg)
        _, _, NZ = cfg.grid
        s = cfg.out_stride
        # 3D stem: xy stride `s` once, z: NZ -> NZ/4
        self.stem = nn.Sequential(
            c3(self.vfe.out_ch, 32, (1, s, s)), c3(32, 32),
            c3(32, 64, (2, 1, 1)), c3(64, 64),
            c3(64, 64, (2, 1, 1)))
        zc = (NZ + 3) // 4
        Cb = cfg.bev_ch
        self.to_bev = c2(64 * zc, Cb)
        # 2D encoder (3 scales) + SECOND-style decoder
        self.e1 = nn.Sequential(c2(Cb, Cb), c2(Cb, Cb))
        self.e2 = nn.Sequential(c2(Cb, 2 * Cb, 2), c2(2 * Cb, 2 * Cb), c2(2 * Cb, 2 * Cb))
        self.e3 = nn.Sequential(c2(2 * Cb, 2 * Cb, 2), c2(2 * Cb, 2 * Cb), c2(2 * Cb, 2 * Cb))
        self.d1 = c2(Cb, Cb)
        self.d2 = up2(2 * Cb, Cb, 2)
        self.d3 = up2(2 * Cb, Cb, 4)
        self.fuse = nn.Sequential(c2(3 * Cb, 192), c2(192, 192))
        self.heads = nn.ModuleDict({k: nn.Sequential(c2(192, 64), nn.Conv2d(64, n, 1)) for k, n in HEADS.items()})
        nn.init.constant_(self.heads["hm"][-1].bias, -2.19)   # prior p=0.1 -> stable focal-loss start

    def forward(self, pts, pmsk):
        with torch.autocast(device_type=pts.device.type, enabled=False):
            v = self.vfe(pts.float(), pmsk)            # scatter in fp32
        x = self.stem(v)                               # (B,64,zc,NXc,NYc)
        B, C, Z, H, W = x.shape
        x = self.to_bev(x.reshape(B, C * Z, H, W))
        x1 = self.e1(x); x2 = self.e2(x1); x3 = self.e3(x2)
        f = self.fuse(torch.cat([self.d1(x1), self.d2(x2), self.d3(x3)], 1))
        return {k: h(f) for k, h in self.heads.items()}


Writing /kaggle/working/code/d3/model.py


In [14]:
%%writefile /kaggle/working/code/d3/post.py
"""
Post-processing on saved raw detections -- no learning, only hand-designed rules
whose few knobs are tuned on val with the official scorer (allowed by the rules).

Pipeline (each step can be switched off by its parameter):
  1. fuse      : merge several models / mirror-TTA into one set of boxes per frame
  2. flip      : head-down predictions (tilt > flip_tilt) are turned head-up (R @ Rx(pi)).
                 Val has 1.2% head-down divers; head<->feet confusion costs 180 deg.
  3. track     : link detections across consecutive frames (greedy, BEV distance gate)
  4. smooth    : per track, Gaussian-weighted window over neighbouring frames for
                 center (sig_c), rotation (sig_r, chordal mean after symmetry alignment
                 and local head-direction voting) and dims (sig_d).
  5. rescore   : blend each score with its track's mean; penalise short tracks
                 (one-frame blips are usually false positives, e.g. in empty frames).
  6. dim_scale : multiply dims per axis (corrects systematic size bias -> ASE).
  7. final threshold / max detections per frame.
Frames are scored independently, so using past AND future frames (offline smoothing)
only uses the test point clouds themselves -- no labels, nothing learned.
"""
import numpy as np
from .geometry import Rx, SYM, project_to_SO3, rot_angle_deg, tilt_deg

DEFAULTS = dict(
    fuse_dist=0.6, flip_tilt=999.0,
    track_thr=0.05, gate=0.8, max_age=3,
    sig_c=0.0, sig_r=0.0, sig_d=0.0,
    flip_vote=0,                 # 1 = within a window, flip a box whose head disagrees with the score-weighted majority
    a_score=0.0, min_len=1, short_pen=1.0,
    dim_scale=(1.0, 1.0, 1.0),
    final_thr=0.0, max_det=20,
)


def _empty():
    return dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)), R=np.zeros((0, 3, 3)), score=np.zeros(0))


def scene_of(tok):
    return tok.split("/")[0]


def frame_of(tok):
    return int(tok.split("/")[1])


# ----------------------------------------------------------------------------- 1. fusion
def expand_tta(p):
    """Mirror-TTA predictions are stored as {'a': orig, 'm': mirrored}; split them."""
    return [p["a"], p["m"]] if "a" in p else [p]


def fuse_frame(plist, weights, dist):
    """plist: list over models of dicts. Greedy clustering by BEV distance; at most one
    box per model per cluster. Fused score = sum_m w_m s_m / sum_m w_m (missing = 0)."""
    items = []
    for m, (p, w) in enumerate(zip(plist, weights)):
        for k in range(len(p["score"])):
            items.append((p["score"][k] * w, m, p["c"][k], p["d"][k], p["R"][k], p["score"][k]))
    if not items:
        return _empty()
    items.sort(key=lambda t: -t[0])
    clusters = []
    for it in items:
        for cl in clusters:
            if it[1] not in cl["models"] and np.hypot(*(it[2][:2] - cl["c0"][:2])) < dist:
                cl["items"].append(it); cl["models"].add(it[1]); break
        else:
            clusters.append(dict(c0=it[2], items=[it], models={it[1]}))
    W = float(sum(weights))
    out = _empty()
    C, D, Rs, S = [], [], [], []
    for cl in clusters:
        ws = np.array([t[0] for t in cl["items"]]) + 1e-9
        C.append((ws[:, None] * np.array([t[2] for t in cl["items"]])).sum(0) / ws.sum())
        D.append((ws[:, None] * np.array([t[3] for t in cl["items"]])).sum(0) / ws.sum())
        Rref = cl["items"][0][4]
        acc = np.zeros((3, 3))
        for t, w in zip(cl["items"], ws):
            R = t[4]
            if rot_angle_deg(Rref, R @ SYM) < rot_angle_deg(Rref, R):
                R = R @ SYM
            acc += w * R
        Rs.append(project_to_SO3(acc))
        S.append(sum(t[0] for t in cl["items"]) / W)
    return dict(c=np.array(C), d=np.array(D), R=np.array(Rs), score=np.array(S))


def fuse(models, weights=None, dist=0.6):
    """models: list of {token: pred}. Mirror-TTA entries count as two equal members."""
    weights = weights or [1.0] * len(models)
    toks = set().union(*[m.keys() for m in models])
    out = {}
    for t in toks:
        plist, wl = [], []
        for m, w in zip(models, weights):
            if t in m:
                parts = expand_tta(m[t])
                plist += parts; wl += [w / len(parts)] * len(parts)
            else:
                plist.append(_empty()); wl.append(w)
        out[t] = plist[0] if len(plist) == 1 else fuse_frame(plist, wl, dist)
    return out


# ----------------------------------------------------------------------------- 3. tracking
def track_scene(frames, preds, P):
    """frames: tokens of ONE scene in order. Returns list of tracks; each track is a
    list of (frame_index, token, det_index)."""
    tracks, active = [], []          # active: dict(id, c, v, f)
    for fi, tok in enumerate(frames):
        p = preds[tok]
        order = np.argsort(-p["score"])
        used = set()
        for k in order:
            if p["score"][k] < P["track_thr"]:
                continue
            c = p["c"][k]
            best, bd = None, P["gate"]
            for a in active:
                if a["id"] in used or fi - a["f"] > P["max_age"]:
                    continue
                pred_c = a["c"] + a["v"] * (fi - a["f"])
                dd = np.hypot(*(c[:2] - pred_c[:2]))
                if dd < bd:
                    best, bd = a, dd
            if best is None:
                best = dict(id=len(tracks), c=c.copy(), v=np.zeros(3), f=fi)
                tracks.append([]); active.append(best)
            else:
                # alpha-beta filter: smooth position AND velocity, so one noisy detection
                # does not throw the next prediction outside the gate (track fragmentation)
                gap = fi - best["f"]
                xp = best["c"] + best["v"] * gap
                res = c - xp
                best["c"] = xp + 0.5 * res
                v = best["v"] + 0.1 * res / gap
                sp = np.linalg.norm(v[:2])
                best["v"] = v * min(1.0, 0.4 / sp) if sp > 0 else v      # divers: p99 0.6 m/frame
                best["f"] = fi
            used.add(best["id"])
            tracks[best["id"]].append((fi, tok, k))
        active = [a for a in active if fi - a["f"] <= P["max_age"]]
    return tracks


def _gauss(dt, sig):
    return np.exp(-0.5 * (dt / sig) ** 2) if sig > 0 else (dt == 0).astype(float)


# ----------------------------------------------------------------------------- main entry
def postprocess(preds, P=None):
    """preds: {token: dict(c,d,R,score)} (single model or already fused). Returns new dict."""
    P = {**DEFAULTS, **(P or {})}
    out = {t: {k: v.copy() for k, v in p.items()} for t, p in preds.items()}

    # 2. head-down -> head-up
    if P["flip_tilt"] < 180:
        F = Rx(np.pi)
        for p in out.values():
            if len(p["R"]):
                m = tilt_deg(p["R"]) > P["flip_tilt"]
                p["R"][m] = p["R"][m] @ F

    smooth = P["sig_c"] > 0 or P["sig_r"] > 0 or P["sig_d"] > 0 or P["a_score"] > 0 \
        or P["min_len"] > 1 or P["flip_vote"]
    if smooth:
        scenes = {}
        for t in out:
            scenes.setdefault(scene_of(t), []).append(t)
        for sc, toks in scenes.items():
            toks.sort(key=frame_of)
            src = {t: out[t] for t in toks}
            snap = {t: {k: v.copy() for k, v in out[t].items()} for t in toks}   # read from snapshot
            for tr in track_scene(toks, src, P):
                fi = np.array([x[0] for x in tr], float)
                C = np.array([snap[t]["c"][k] for _, t, k in tr])
                D = np.array([snap[t]["d"][k] for _, t, k in tr])
                R = np.array([snap[t]["R"][k] for _, t, k in tr])
                S = np.array([snap[t]["score"][k] for _, t, k in tr])
                n = len(tr)
                RS = R @ SYM                                  # symmetric twins, precomputed
                smax = max(P["sig_c"], P["sig_d"], P["sig_r"], 2.0 if P["flip_vote"] else 0.0)
                half = int(np.ceil(3 * smax))                 # Gaussian is ~0 beyond 3 sigma
                for j, (_, t, k) in enumerate(tr):
                    lo = np.searchsorted(fi, fi[j] - half, "left")
                    hi = np.searchsorted(fi, fi[j] + half, "right")
                    dt, Sw = fi[lo:hi] - fi[j], S[lo:hi]
                    if P["sig_c"] > 0:
                        w = _gauss(dt, P["sig_c"]) * Sw
                        out[t]["c"][k] = (w[:, None] * C[lo:hi]).sum(0) / w.sum()
                    if P["sig_d"] > 0:
                        w = _gauss(dt, P["sig_d"]) * Sw
                        out[t]["d"][k] = (w[:, None] * D[lo:hi]).sum(0) / w.sum()
                    if P["sig_r"] > 0 or P["flip_vote"]:
                        Rj = R[j]
                        if P["flip_vote"]:
                            w = _gauss(dt, max(P["sig_r"], 2.0)) * Sw
                            if Rj[:, 2] @ (w[:, None] * R[lo:hi, :, 2]).sum(0) < 0:
                                Rj = Rj @ Rx(np.pi)          # local majority says head is the other way
                        if P["sig_r"] > 0:
                            w = _gauss(dt, P["sig_r"]) * Sw
                            Rw, RSw = R[lo:hi], RS[lo:hi]
                            w = w * (Rw[:, :, 2] @ Rj[:, 2] >= 0)                  # skip head-disagreeing
                            # pick R or its Rz(pi) twin, whichever is closer to Rj (larger trace)
                            tr1 = np.einsum("ab,nab->n", Rj, Rw)
                            tr2 = np.einsum("ab,nab->n", Rj, RSw)
                            Ral = np.where((tr2 > tr1)[:, None, None], RSw, Rw)
                            acc = (w[:, None, None] * Ral).sum(0)
                            if np.abs(acc).sum() > 0:
                                Rj = project_to_SO3(acc)
                        out[t]["R"][k] = Rj
                    if P["a_score"] > 0:
                        out[t]["score"][k] = S[j] ** (1 - P["a_score"]) * S.mean() ** P["a_score"]
                    if n < P["min_len"]:
                        out[t]["score"][k] *= P["short_pen"]

    ds = np.array(P["dim_scale"], float)
    for t, p in out.items():
        if len(p["d"]):
            p["d"] = p["d"] * ds
        keep = p["score"] >= P["final_thr"]
        p2 = {k: v[keep] for k, v in p.items()}
        order = np.argsort(-p2["score"])[:P["max_det"]]
        out[t] = {k: v[order] for k, v in p2.items()}
    return out


Writing /kaggle/working/code/d3/post.py


In [15]:
%%writefile /kaggle/working/code/d3/predict.py
"""
Predict with a checkpoint on val and/or test; save raw detections (for later
post-processing / ensembling) and a ready-to-validate submission file.

  python -m d3.predict --root ROOT --ckpt runs/A/best.pt --out runs/A --splits val test [--mirror]

--mirror : also run the y-mirrored point cloud and un-mirror its boxes (test-time
           augmentation). Saves raw_{split}_tta.pkl.gz = {token: {"a": orig, "m": mirrored}}
           (fused later by d3.post.fuse) AND the plain raw_{split}.pkl.gz from the same pass.
"""
import argparse, json, os, subprocess, sys
import torch
from .config import Cfg
from .model import VoxelCenterNet
from .infer import run_inference, save_preds
from .io import write_submission
from .evaluate import run_scorer, fmt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--splits", nargs="+", default=["val", "test"])
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--thr", type=float, default=0.03)
    ap.add_argument("--mirror", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    dev = f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu"
    ck = torch.load(args.ckpt, map_location=dev, weights_only=False)
    cfg = Cfg.from_dict(ck["cfg"])
    model = VoxelCenterNet(cfg).to(dev)
    model.load_state_dict(ck["model"])
    for split in args.splits:
        if split == "test":
            tokens = [l.strip() for l in open(os.path.join(args.root, "annotations", "test_tokens.txt")) if l.strip()]
        else:
            tokens = list(json.load(open(os.path.join(args.root, "annotations", f"gt_{split}.json")))["results"])
        preds = run_inference(model, args.root, tokens, cfg, device=dev, thr=args.thr, mirror=args.mirror)
        if args.mirror:
            save_preds(preds, os.path.join(args.out, f"raw_{split}_tta.pkl.gz"))
            preds = {t: p["a"] for t, p in preds.items()}
        save_preds(preds, os.path.join(args.out, f"raw_{split}.pkl.gz"))
        sub = write_submission(preds, os.path.join(args.out, f"sub_{split}.json"), tokens)
        if split == "val":
            print("VAL", args.out, fmt(run_scorer(args.root, sub)), flush=True)
        else:
            r = subprocess.run([sys.executable, os.path.join(args.root, "tools", "validate_submission.py"),
                                "--pred", sub, "--tokens",
                                os.path.join(args.root, "annotations", "test_tokens.txt")],
                               capture_output=True, text=True)
            print("validate_submission:", (r.stdout + r.stderr)[-400:], flush=True)


if __name__ == "__main__":
    main()


Writing /kaggle/working/code/d3/predict.py


In [16]:
%%writefile /kaggle/working/code/d3/rot_torch.py
"""
Torch rotation utilities for the network head and the loss.

Representation: the head outputs 6 numbers (a, b) ~ (ez, ex); Gram-Schmidt
turns them into R = [ex, ey, ez]. Identical to geometry.rot6d_zx_to_R.

Loss: symmetric CHORDAL loss, matching the metric's symmetry
    L = min( ||Rp - Rg||_F^2 , ||Rp - Rg S||_F^2 ),   S = diag(-1,-1,1)
Chordal ||A-B||^2 = 8 sin^2(theta/2) is monotonic in the geodesic angle
theta, so it has the same minimiser and the same choice of S. Unlike acos(),
its gradient is smooth everywhere (acos blows up near 0 deg and is zero after
clamping, which is why the old rot loss jumped around).
"""
import torch
import torch.nn.functional as F


def rot6d_zx_to_R(r6):
    """(K,6) -> (K,3,3) columns [ex, ey, ez]."""
    ez = F.normalize(r6[:, :3], dim=1, eps=1e-6)
    b = r6[:, 3:]
    ex = F.normalize(b - (ez * b).sum(1, keepdim=True) * ez, dim=1, eps=1e-6)
    ey = torch.cross(ez, ex, dim=1)
    return torch.stack([ex, ey, ez], dim=2)


def sym_chordal_loss(Rp, Rg):
    """(K,3,3),(K,3,3) -> (K,) per-box loss, in [0, 8]."""
    d_id = ((Rp - Rg) ** 2).sum((1, 2))
    sgn = Rg.new_tensor([-1.0, -1.0, 1.0])          # Rg @ S negates columns ex, ey
    d_sym = ((Rp - Rg * sgn) ** 2).sum((1, 2))
    return torch.minimum(d_id, d_sym)


def sym_angle_deg(Rp, Rg):
    """For logging only (not for backprop)."""
    def ang(A, B):
        tr = (A.transpose(1, 2) @ B).diagonal(dim1=1, dim2=2).sum(-1)
        return torch.rad2deg(torch.acos(((tr - 1) / 2).clamp(-1, 1)))
    sgn = Rg.new_tensor([-1.0, -1.0, 1.0])
    return torch.minimum(ang(Rp, Rg), ang(Rp, Rg * sgn))


Writing /kaggle/working/code/d3/rot_torch.py


In [17]:
%%writefile /kaggle/working/code/d3/train.py
"""
Resumable training.

  python -m d3.train --root ROOT --out /kaggle/working/runs/v1 --gpu 0 --set epochs=20 rebalance=0.5

Every epoch writes OUT/last.pt (model, EMA, optimizer, scheduler, scaler, epoch).
If OUT/last.pt exists the run RESUMES from it automatically; --init PATH copies a
checkpoint from elsewhere (e.g. a previous Kaggle version's output) first.
Validation uses the EMA weights + the OFFICIAL scorer; best checkpoint -> OUT/best.pt.

  --overfit N : sanity mode. Train on N val-free train frames without augmentation and
                score on those same frames. Should reach SDS > 0.9 in a few hundred
                iterations; if not, there is a bug in targets/decoding/conventions.
"""
import argparse, copy, json, math, os, random, shutil, sys, time
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from .config import Cfg
from .data import DiverDataset, GTDatabase, collate
from .model import VoxelCenterNet
from .losses import compute_loss
from .infer import run_inference, save_preds
from .io import load_gt, write_submission
from .evaluate import run_scorer, fmt


class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval()
        for p in self.m.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))      # warm-up
        msd = model.state_dict()
        for k, v in self.m.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


def log(out, msg):
    print(msg, flush=True)
    with open(os.path.join(out, "log.txt"), "a") as f:
        f.write(msg + "\n")


def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def evaluate(model, root, gt_path, tokens, cfg, out, tag, device):
    preds = run_inference(model, root, tokens, cfg, device=device, workers=cfg.workers)
    save_preds(preds, os.path.join(out, f"preds_val_{tag}.pkl.gz"))
    p = write_submission(preds, os.path.join(out, f"pred_val_{tag}.json"), tokens)
    m = run_scorer(root, p, gt_path)
    os.remove(p)
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--set", nargs="*", default=[])
    ap.add_argument("--init", default=None, help="checkpoint to resume from if OUT has none")
    ap.add_argument("--overfit", type=int, default=0)
    ap.add_argument("--max_iters", type=int, default=0, help="debug: stop early")
    ap.add_argument("--cpu", action="store_true")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    device = "cpu" if args.cpu else f"cuda:{args.gpu}"
    dev_type = "cpu" if args.cpu else "cuda"
    cfg = Cfg().update(args.set)
    if args.overfit:
        cfg.update(["gt_sample=False", "mirror_p=0.0", "rot_deg=0.0", "scale_lo=1.0", "scale_hi=1.0",
                    "drop_max=0.0", "int_jitter=0.0", "ema=0.0"])
    json.dump(cfg.to_dict(), open(os.path.join(args.out, "cfg.json"), "w"), indent=1)
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    torch.backends.cudnn.benchmark = True

    gt_train = load_gt(os.path.join(args.root, "annotations", "gt_train.json"))
    gt_val_path = os.path.join(args.root, "annotations", "gt_val.json")
    if args.overfit:
        toks = [t for t, v in gt_train.items() if len(v["c"])][:: max(1, len(gt_train) // (3 * args.overfit))][:args.overfit]
        gt_train = {t: gt_train[t] for t in toks}
        raw = json.load(open(os.path.join(args.root, "annotations", "gt_train.json")))["results"]
        gt_val_path = os.path.join(args.out, "gt_overfit.json")
        json.dump({"split": "train", "results": {t: raw[t] for t in toks}}, open(gt_val_path, "w"))
        val_tokens = toks
    else:
        val_tokens = list(json.load(open(gt_val_path))["results"].keys())

    db = None
    if cfg.gt_sample:
        t0 = time.time()
        db = GTDatabase(args.root, gt_train, os.path.join(os.path.dirname(args.out.rstrip("/")), "gtdb.npz"),
                        margin=cfg.gt_margin)
        log(args.out, f"GT database: {len(db)} divers ({time.time()-t0:.0f}s)")
    ds = DiverDataset(args.root, gt_train, cfg, train=True, db=db)
    if args.overfit:
        ds.tokens = ds.tokens * max(1, 2000 // len(ds.tokens))
    if cfg.rebalance > 0:
        sampler = WeightedRandomSampler(torch.as_tensor(ds.frame_weights()), len(ds), replacement=True,
                                        generator=torch.Generator().manual_seed(cfg.seed))
    else:
        sampler = None
    def make_loader(epoch):
        if sampler is not None:
            sampler.generator.manual_seed(cfg.seed * 1000 + epoch)
        g = torch.Generator().manual_seed(cfg.seed * 1000 + epoch)
        return DataLoader(ds, batch_size=cfg.bs, shuffle=sampler is None, sampler=sampler, generator=g,
                          num_workers=cfg.workers, collate_fn=collate, drop_last=True, pin_memory=not args.cpu)

    model = VoxelCenterNet(cfg).to(device)
    log(args.out, f"params {sum(p.numel() for p in model.parameters())/1e6:.2f}M | grid {cfg.grid} bev {cfg.bev}")
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    steps_per_epoch = len(ds) // cfg.bs
    total = cfg.epochs * steps_per_epoch
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=total, pct_start=0.3,
                                                div_factor=10, final_div_factor=100)
    scaler = torch.amp.GradScaler(dev_type, enabled=not args.cpu)
    ema = EMA(model, cfg.ema) if cfg.ema > 0 else None
    start_epoch, best = 0, -1.0

    last = os.path.join(args.out, "last.pt")
    if not os.path.exists(last) and args.init and os.path.exists(args.init):
        shutil.copy(args.init, last)
        log(args.out, f"copied init checkpoint {args.init}")
    if os.path.exists(last):
        ck = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        if ema is not None and ck.get("ema"):
            ema.m.load_state_dict(ck["ema"]); ema.n = ck["ema_n"]
        start_epoch, best = ck["epoch"] + 1, ck.get("best", -1.0)
        log(args.out, f"RESUMED at epoch {start_epoch} (best SDS so far {best:.4f})")

    it_global = start_epoch * steps_per_epoch
    for epoch in range(start_epoch, cfg.epochs):
        ds.set_epoch(epoch)
        model.train()
        t0, agg, nb = time.time(), {}, 0
        for b in make_loader(epoch):
            pts, pmsk = b["pts"].to(device, non_blocking=True), b["pmsk"].to(device, non_blocking=True)
            batch = {k: b[k].to(device, non_blocking=True) for k in ("hm", "ind", "msk", "reg")}
            with torch.autocast(device_type=dev_type, dtype=torch.float16, enabled=not args.cpu):
                out = model(pts, pmsk)
            loss, parts = compute_loss(out, batch, cfg)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip)
            scaler.step(opt); scaler.update()
            if it_global < total:
                sched.step()
            if ema is not None:
                ema.update(model)
            it_global += 1; nb += 1
            for k, v in parts.items():
                agg[k] = agg.get(k, 0) + v
            if nb % 200 == 0 or nb == 20:
                log(args.out, f"ep {epoch} it {nb}/{steps_per_epoch} loss {loss.item():.3f} "
                    + " ".join(f"{k} {v/nb:.3f}" for k, v in agg.items())
                    + f" | {(time.time()-t0)/nb:.3f}s/it lr {sched.get_last_lr()[0]:.2e}"
                    + (f" mem {torch.cuda.max_memory_allocated(device)/1e9:.1f}GB" if not args.cpu else ""))
            if args.max_iters and nb >= args.max_iters:
                break
        log(args.out, f"== epoch {epoch} done in {(time.time()-t0)/60:.1f} min | "
            + " ".join(f"{k} {v/max(nb,1):.3f}" for k, v in agg.items()))

        ck = dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                  scaler=scaler.state_dict(), ema=ema.m.state_dict() if ema else None,
                  ema_n=ema.n if ema else 0, epoch=epoch, best=best, cfg=cfg.to_dict())
        atomic_save(ck, last)

        is_last3 = epoch >= cfg.epochs - 3
        if (epoch + 1) % cfg.eval_every == 0 or is_last3 or args.overfit:
            net = ema.m if ema is not None else model
            m = evaluate(net, args.root, gt_val_path, val_tokens, cfg, args.out, f"ep{epoch:02d}", device)
            log(args.out, f"** VAL epoch {epoch}: {fmt(m)}")
            if m["SDS"] > best:
                best = m["SDS"]
                ck["best"] = best
                atomic_save(dict(model=net.state_dict(), cfg=cfg.to_dict(), epoch=epoch, metrics=m),
                            os.path.join(args.out, "best.pt"))
                atomic_save(ck, last)
                log(args.out, f"   new best SDS {best:.4f}")
    log(args.out, f"DONE. best val SDS {best:.4f}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/code/d3/train.py


In [18]:
%%writefile /kaggle/working/code/d3/transforms.py
"""
Joint point/box transforms used by augmentation and test-time augmentation (TTA).

Boxes are kept as arrays:  c (K,3) centers, d (K,3) dims [l,w,h], R (K,3,3).
Each function returns NEW arrays (never modifies inputs in place).

Rule for a proper rotation T (det +1) applied to the whole scene:
    p' = T p,  c' = T c,  R' = T R,  d' = d
Rule for the mirror M = diag(1,-1,1) (det -1, a reflection):
    p' = M p,  c' = M c,  R' = M R M,  d' = d
  Why M R M and not M R?  M R has det -1 (not a rotation). Mirroring the body
  swaps its left and right shoulders, so the new ey (right->left shoulder)
  is -(M ey). Negating the 2nd column is exactly right-multiplying by M:
    R' = [M ex, -M ey, M ez] = M R M,  and  ex' x ey' = M ez = ez'.
"""
import numpy as np
from .geometry import MIRROR_Y, Rz


def mirror_y(pts, c, R):
    pts = pts.copy(); pts[:, 1] = -pts[:, 1]
    c = c * np.array([1.0, -1.0, 1.0])
    R = MIRROR_Y @ R @ MIRROR_Y
    return pts, c, R


def rotate_z(pts, c, R, theta):
    """Rotate the scene about the SONAR z-axis through the sonar origin.
    This keeps each point's range and elevation and only changes bearing,
    so the sonar's view of the diver stays physically plausible."""
    T = Rz(theta)
    pts = pts.copy(); pts[:, :3] = pts[:, :3] @ T.T
    return pts, c @ T.T, T @ R


def scale(pts, c, d, s):
    pts = pts.copy(); pts[:, :3] *= s
    return pts, c * s, d * s


def mirror_boxes_only(c, R):
    """Undo a mirror on PREDICTIONS (for mirror TTA): the mirror is its own inverse."""
    return c * np.array([1.0, -1.0, 1.0]), MIRROR_Y @ R @ MIRROR_Y


Writing /kaggle/working/code/d3/transforms.py


In [19]:
%%writefile /kaggle/working/code/tests/check_with_scorer.py
"""
Use the OFFICIAL scorer as the oracle for our conventions, and measure how
sensitive the metric is to each kind of error. Needs the dataset.

    python tests/check_with_scorer.py --root /kaggle/input/<...>/diver3d_v2

Part A (must pass): perfect GT written through OUR reader/writer scores 1.0, the
Rz(pi) twin is free, the head<->feet flip costs exactly 180 deg, and a simulated
quaternion-order bug is caught.

Part B (for understanding): perturb the perfect GT by one error type at a time
and watch what the metric does. This tells us which errors to fight.
"""
import argparse, os, sys, copy
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.io import load_gt, write_submission
from d3.geometry import SYM, Rx, Rz, quat_to_R, R_to_quat
from d3 import transforms as T
from d3.evaluate import run_scorer, fmt

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
ap.add_argument("--out", default="/kaggle/working/scorer_checks")
ap.add_argument("--skip_sensitivity", action="store_true")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)

gt = load_gt(os.path.join(args.root, "annotations", "gt_val.json"))
tokens = list(gt.keys())


def score(name, fn):
    p = {t: fn(copy.deepcopy(v)) for t, v in gt.items()}
    path = write_submission(p, os.path.join(args.out, name + ".json"), tokens)
    m = run_scorer(args.root, path)
    print(f"{name:28s} {fmt(m)}")
    return m


def apply_R(f):
    def g(b):
        if len(b["R"]): b["R"] = f(b["R"])
        return b
    return g


print("== Part A: convention checks against the official scorer ==")
m = score("identity", lambda b: b)
assert abs(m["SDS"] - 1) < 1e-3, "our reader/writer is broken"
m = score("rz_pi_twin", apply_R(lambda R: R @ SYM))
assert m["AOE3D_deg"] < 0.01 and abs(m["SDS"] - 1) < 1e-3, "Rz(pi) should be free"
m = score("head_feet_flip", apply_R(lambda R: R @ Rx(np.pi)))
assert abs(m["AOE3D_deg"] - 180) < 0.1, "head flip should cost 180 deg"
m = score("quat_order_bug_xyzw", apply_R(lambda R: quat_to_R(R_to_quat(R)[:, [1, 2, 3, 0]])))
assert m["SDS"] < 0.9, "scorer should punish wrong quaternion order"


def mirror_twice(b):
    if len(b["c"]):
        _, c, R = T.mirror_y(np.zeros((1, 4)), b["c"], b["R"])
        _, b["c"], b["R"] = T.mirror_y(np.zeros((1, 4)), c, R)
    return b
m = score("mirror_twice", mirror_twice)
assert abs(m["SDS"] - 1) < 1e-3
print("Part A passed: conventions agree with the official scorer.\n")

if args.skip_sensitivity:
    sys.exit()

print("== Part B: metric sensitivity (one error type at a time) ==")
def shift(axis, delta):
    def f(b):
        if len(b["c"]): b["c"][:, axis] += delta
        return b
    return f
def scale_dim(k, s):
    def f(b):
        if len(b["d"]): b["d"][:, k] *= s
        return b
    return f
for dz in (0.1, 0.2, 0.3):
    score(f"z +{dz} m", shift(2, dz))
for dx in (0.1, 0.2, 0.3):
    score(f"x +{dx} m (range)", shift(0, dx))
for s in (0.85, 1.15):
    score(f"height x{s}", scale_dim(2, s))
    score(f"length x{s}", scale_dim(0, s))
for deg in (10, 20, 30):
    score(f"tilt about ex {deg} deg", apply_R(lambda R, a=np.radians(deg): R @ Rx(a)))
    score(f"yaw about ez {deg} deg", apply_R(lambda R, a=np.radians(deg): R @ Rz(a)))


Writing /kaggle/working/code/tests/check_with_scorer.py


In [20]:
%%writefile /kaggle/working/code/tests/make_mock.py
"""Synthetic mini-dataset with the real file layout, for CPU smoke tests.
Divers are dense box-shaped blobs; background is uniform clutter.
Also writes a crude stand-in scorer (fraction of GT matched within 0.4 m) so
train.py can run end-to-end without the real tools/diver3d_eval.py."""
import json, os, sys
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.geometry import quat_to_R, R_to_quat

MOCK_SCORER = r'''
import argparse, json, numpy as np
ap = argparse.ArgumentParser(); ap.add_argument("--gt"); ap.add_argument("--pred"); a = ap.parse_args()
g = json.load(open(a.gt))["results"]; p = json.load(open(a.pred))["results"]
tp = n = 0
for t, bs in g.items():
    n += len(bs)
    P = [b for b in p.get(t, []) if b["score"] > 0.3]
    for b in bs:
        if any(np.linalg.norm(np.subtract(b["center"], q["center"])) < 0.4 for q in P): tp += 1
r = tp / max(n, 1)
print(f"\n  SDS         {r:.4f}\n  mAP3D       {r:.4f}\n")
'''


def make(root, n_scenes=3, n_frames=40, seed=0):
    rng = np.random.default_rng(seed)
    os.makedirs(os.path.join(root, "annotations"), exist_ok=True)
    os.makedirs(os.path.join(root, "tools"), exist_ok=True)
    open(os.path.join(root, "tools", "diver3d_eval.py"), "w").write(MOCK_SCORER)
    for split, off in (("train", 0), ("val", 50)):
        res = {}
        for s in range(n_scenes):
            sc = f"scene_{off + s:04d}"
            os.makedirs(os.path.join(root, "scenes", sc, "sonar"), exist_ok=True)
            k = int(rng.integers(0, 3))
            divers = [(rng.uniform([2, -3, -1], [8, 3, 1]), quat_to_R(rng.normal(size=4))) for _ in range(k)]
            for f in range(n_frames):
                bg = np.c_[rng.uniform([0.2, -8, -4], [15, 8, 4], size=(1500, 3)), rng.uniform(0, 0.3, 1500)]
                boxes, pts = [], [bg]
                for i, (c, R) in enumerate(divers):
                    d = np.array([0.9, 0.82, 1.76])
                    loc = rng.uniform(-0.5, 0.5, size=(600, 3)) * d
                    pts.append(np.c_[loc @ R.T + c, rng.uniform(0.6, 1.0, 600)])
                    boxes.append(dict(center=c.tolist(), dimensions=d.tolist(),
                                      quaternion=R_to_quat(R).tolist(), instance_id=i + 1))
                np.concatenate(pts).astype(np.float32).tofile(
                    os.path.join(root, "scenes", sc, "sonar", f"frame_{f:06d}.bin"))
                res[f"{sc}/{f:06d}"] = boxes
        json.dump({"split": split, "results": res},
                  open(os.path.join(root, "annotations", f"gt_{split}.json"), "w"))
    return root


if __name__ == "__main__":
    make(sys.argv[1])


Writing /kaggle/working/code/tests/make_mock.py


In [21]:
%%writefile /kaggle/working/code/tests/test_data.py
"""
Data/target/decoder/model-layout tests. Run:  python -m pytest -q tests/test_data.py
Uses a synthetic dataset (tests/make_mock.py), no real data needed.
"""
import os, sys, tempfile
import numpy as np
import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.config import Cfg
from d3.data import build_targets, DiverDataset, GTDatabase, collate, gaussian_radius, pose_weight
from d3.decode import decode
from d3.geometry import quat_to_R, sym_angle_deg, points_in_box, tilt_deg
from d3.io import load_gt
from d3.model import VoxelCenterNet, VFE
from make_mock import make

rng = np.random.default_rng(0)
cfg = Cfg()


def random_boxes(k):
    c = np.c_[rng.uniform(1, 12, k), rng.uniform(-6, 6, k), rng.uniform(-2, 2, k)]
    d = rng.uniform([0.6, 0.6, 1.2], [1.2, 1.4, 2.1], size=(k, 3))
    return c, d, quat_to_R(rng.normal(size=(k, 4)))


def perfect_outputs(c, d, R):
    """What a perfect network would output for these boxes."""
    hm, ind, msk, reg = build_targets(c, d, R, cfg)
    NXc, NYc = cfg.bev
    out = {"hm": torch.full((1, 1, NXc, NYc), -20.0),
           "off": torch.zeros(1, 2, NXc, NYc), "z": torch.zeros(1, 1, NXc, NYc),
           "dim": torch.zeros(1, 3, NXc, NYc), "rot": torch.zeros(1, 6, NXc, NYc)}
    for k in range(int(msk.sum())):
        ix, iy = divmod(int(ind[k]), NYc)
        out["hm"][0, 0, ix, iy] = 20.0
        out["off"][0, :, ix, iy] = torch.tensor(reg[k, 0:2])
        out["z"][0, :, ix, iy] = torch.tensor(reg[k, 2:3])
        out["dim"][0, :, ix, iy] = torch.tensor(reg[k, 3:6])
        out["rot"][0, :, ix, iy] = torch.tensor(reg[k, 6:12])
    return out, hm


def test_encode_decode_roundtrip():
    for trial in range(30):
        c, d, R = random_boxes(3)
        if min(np.hypot(*(c[i, :2] - c[j, :2])) for i in range(3) for j in range(i)) < 1.0:
            continue                                    # same/adjacent cell: not a valid scene
        out, _ = perfect_outputs(c, d, R)
        p = decode(out, cfg)[0]
        assert len(p["c"]) == 3
        for i in range(3):
            j = np.argmin(np.linalg.norm(p["c"] - c[i], axis=1))
            assert np.allclose(p["c"][j], c[i], atol=1e-4)
            assert np.allclose(p["d"][j], d[i], atol=1e-4)
            assert sym_angle_deg(p["R"][j], R[i]) < 0.05   # float32 arccos noise


def test_heatmap_peak_at_center_cell():
    c, d, R = random_boxes(1)
    hm, ind, msk, _ = build_targets(c, d, R, cfg)
    ix, iy = divmod(int(ind[0]), cfg.bev[1])
    assert hm[0, ix, iy] == 1.0 and hm.max() == 1.0
    assert int(np.floor((c[0, 0] - cfg.x_range[0]) / cfg.cell)) == ix
    assert int(np.floor((c[0, 1] - cfg.y_range[0]) / cfg.cell)) == iy


def test_gaussian_radius_reasonable():
    r = gaussian_radius(1.0 / 0.2, 1.0 / 0.2, 0.7)   # 1m x 1m footprint in 0.2 m cells
    assert 0.3 < r < 3


def test_vfe_layout():
    """A single point must land in voxel (iz, ix, iy) of the dense grid."""
    vfe = VFE(cfg)
    p = torch.tensor([[[3.05, -1.23, 0.4, 0.5]]])
    v = vfe(p, torch.ones(1, 1, dtype=torch.bool))
    ix = int((3.05 - cfg.x_range[0]) // cfg.vox[0]); iy = int((-1.23 - cfg.y_range[0]) // cfg.vox[1])
    iz = int((0.4 - cfg.z_range[0]) // cfg.vox[2])
    cnt = v[0, -1]                                     # log-count channel, (NZ, NX, NY)
    assert cnt[iz, ix, iy] > 0 and (cnt > 0).sum() == 1


def test_pose_weight():
    assert np.allclose(pose_weight([0, 50, 90, 150], 0), 1)
    w = pose_weight([0, 90], 1)
    assert w[0] > 1 > w[1]


@pytest.fixture(scope="module")
def mock_root():
    return make(tempfile.mkdtemp())


def test_dataset_and_gt_paste(mock_root):
    gt = load_gt(os.path.join(mock_root, "annotations", "gt_train.json"))
    c = Cfg().update(["rebalance=0.5"])
    db = GTDatabase(mock_root, gt, os.path.join(tempfile.mkdtemp(), "db.npz"))
    assert len(db) > 0
    ds = DiverDataset(mock_root, gt, c, train=True, db=db)
    for i in range(0, len(ds), 7):
        item = ds[i]
        n = int(item["msk"].sum())
        assert n == c.gt_target                         # curriculum fills frames up to 4
        # each target box must contain diver points (reconstruct box from targets)
        pts = item["pts"].numpy()
        NYc = c.bev[1]
        for k in range(n):
            ix, iy = divmod(int(item["ind"][k]), NYc)
            r = item["reg"][k].numpy()
            ctr = np.array([c.x_range[0] + (ix + r[0]) * c.cell, c.y_range[0] + (iy + r[1]) * c.cell, r[2]])
            from d3.geometry import rot6d_zx_to_R
            Rk = rot6d_zx_to_R(r[6:9], r[9:12]); dk = np.exp(r[3:6]) * np.array(c.mean_dims)
            assert points_in_box(pts, ctr, dk, Rk).sum() > 100
    # determinism: same (epoch, index) -> identical sample
    a, b = ds[3], ds[3]
    assert torch.equal(a["pts"], b["pts"])


def test_model_forward_shapes():
    c = Cfg()
    m = VoxelCenterNet(c).eval()
    pts = torch.rand(2, 500, 4) * torch.tensor([12, 12, 6, 1]) + torch.tensor([0, -6, -3, 0])
    out = m(pts, torch.ones(2, 500, dtype=torch.bool))
    NXc, NYc = c.bev
    assert out["hm"].shape == (2, 1, NXc, NYc) and out["rot"].shape == (2, 6, NXc, NYc)


Writing /kaggle/working/code/tests/test_data.py


In [22]:
%%writefile /kaggle/working/code/tests/test_geometry.py
"""
Convention unit tests. Run:  python -m pytest -q tests/test_geometry.py
Needs no data. Every test states the fact it checks.
"""
import os, sys, json, tempfile
import numpy as np
import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.geometry import (quat_to_R, R_to_quat, rot6d_zx_to_R, R_to_rot6d_zx, SYM,
                         MIRROR_Y, sym_angle_deg, rot_angle_deg, tilt_deg, Rz, Rx,
                         box_corners, points_in_box, project_to_SO3, align_to_reference)
from d3 import transforms as T
from d3 import rot_torch
from d3.io import write_submission

rng = np.random.default_rng(0)


def rand_R(n):
    q = rng.normal(size=(n, 4))
    return quat_to_R(q)


def rand_box():
    c = rng.uniform([1, -4, -2], [10, 4, 2])
    d = rng.uniform([0.6, 0.6, 1.2], [1.2, 1.4, 2.1])
    return c, d, rand_R(1)[0]


def same_point_set(A, B, tol=1e-9):
    """Corners can come out in a different order; compare as sets."""
    D = np.linalg.norm(A[:, None] - B[None], axis=-1)
    return np.all(D.min(1) < tol) and np.all(D.min(0) < tol)


# ----------------------------------------------------------------- quaternions
def test_quat_order_is_wxyz():
    # 90 deg about z: q = (cos45, 0, 0, sin45). Must send x-axis to y-axis.
    R = quat_to_R([np.cos(np.pi / 4), 0, 0, np.sin(np.pi / 4)])
    assert np.allclose(R[:, 0], [0, 1, 0]) and np.allclose(R[:, 2], [0, 0, 1])


def test_quat_roundtrip_and_sign():
    R = rand_R(1000)
    q = R_to_quat(R)
    assert np.all(q[:, 0] >= 0)                      # canonical sign
    assert np.allclose(quat_to_R(q), R, atol=1e-9)
    assert np.allclose(quat_to_R(-q), R, atol=1e-9)  # q and -q are the same rotation


def test_rotations_are_proper():
    R = rand_R(1000)
    assert np.allclose(np.linalg.det(R), 1)
    assert np.allclose(np.swapaxes(R, 1, 2) @ R, np.eye(3), atol=1e-9)


def test_columns_are_body_axes():
    # ez = ex x ey  (right-handed body frame)
    R = rand_R(100)
    assert np.allclose(np.cross(R[:, :, 0], R[:, :, 1]), R[:, :, 2], atol=1e-9)


# ----------------------------------------------------------------- 6D
def test_rot6d_roundtrip_numpy_and_torch():
    R = rand_R(500)
    r6 = R_to_rot6d_zx(R)
    assert np.allclose(rot6d_zx_to_R(r6[:, :3], r6[:, 3:]), R, atol=1e-9)
    Rt = rot_torch.rot6d_zx_to_R(torch.tensor(r6)).numpy()
    assert np.allclose(Rt, R, atol=1e-9)


def test_rot6d_keeps_ez_exactly():
    # Gram-Schmidt must keep the FIRST vector (ez) direction, only bend ex.
    a, b = rng.normal(size=(100, 3)), rng.normal(size=(100, 3))
    R = rot6d_zx_to_R(a, b)
    assert np.allclose(R[:, :, 2], a / np.linalg.norm(a, axis=1, keepdims=True))
    assert np.allclose(np.linalg.det(R), 1)


# ----------------------------------------------------------------- metric symmetry
def test_symmetry_is_free_headflip_is_not():
    R = rand_R(200)
    assert np.allclose(sym_angle_deg(R @ SYM, R), 0, atol=1e-5)        # Rz(pi): free
    assert np.allclose(sym_angle_deg(R @ Rx(np.pi), R), 180, atol=1e-4)  # head<->feet
    # SYM is Rz(pi) in the body frame: ex,ey flip, ez unchanged
    assert np.allclose((R @ SYM)[:, :, 2], R[:, :, 2])
    assert np.allclose((R @ SYM)[:, :, :2], -R[:, :, :2])


def test_symmetric_box_is_same_cuboid():
    c, d, R = rand_box()
    assert same_point_set(box_corners(c, d, R), box_corners(c, d, R @ SYM))


def test_torch_loss_symmetric_and_matches_angle():
    R = torch.tensor(rand_R(300))
    Rs = R * R.new_tensor([-1.0, -1.0, 1.0])
    assert torch.allclose(rot_torch.sym_chordal_loss(R, Rs), torch.zeros(300, dtype=R.dtype), atol=1e-9)
    R2 = torch.tensor(rand_R(300))
    ang_np = sym_angle_deg(R2.numpy(), R.numpy())
    ang_t = rot_torch.sym_angle_deg(R2, R).numpy()
    assert np.allclose(ang_np, ang_t, atol=1e-4)
    # chordal = 8 sin^2(theta/2) at the symmetric minimum
    L = rot_torch.sym_chordal_loss(R2, R).numpy()
    assert np.allclose(L, 8 * np.sin(np.radians(ang_np) / 2) ** 2, atol=1e-6)


def test_torch_loss_has_gradient_at_zero_error():
    R = torch.tensor(rand_R(4))
    r6 = torch.tensor(R_to_rot6d_zx(R.numpy())) + 1e-3
    r6.requires_grad_(True)
    rot_torch.sym_chordal_loss(rot_torch.rot6d_zx_to_R(r6), R).sum().backward()
    assert torch.isfinite(r6.grad).all()


def test_align_to_reference():
    R = rand_R(50)
    R_ref = R @ SYM @ Rz(0.05)        # close to the symmetric twin
    A = align_to_reference(R, R_ref)
    assert np.all(rot_angle_deg(A, R_ref) < 5)


def test_project_to_SO3():
    R = rand_R(10)
    assert np.allclose(project_to_SO3(R + 1e-3 * rng.normal(size=R.shape)), R, atol=1e-2)
    # chordal mean of a rotation and itself is itself
    assert np.allclose(project_to_SO3((R + R) / 2), R, atol=1e-9)


# ----------------------------------------------------------------- augmentation transforms
def scene_with_box(n=4000):
    c, d, R = rand_box()
    pts = np.c_[rng.uniform([0, -5, -2.5], [12, 5, 2.5], size=(n, 3)), rng.uniform(size=n)]
    inside = (rng.uniform(-0.5, 0.5, size=(500, 3)) * d) @ R.T + c   # points inside the box
    pts = np.r_[pts, np.c_[inside, rng.uniform(size=500)]]
    return pts, c[None], d[None], R[None]


@pytest.mark.parametrize("which", ["mirror", "rotz", "scale"])
def test_transform_moves_box_with_points(which):
    pts, c, d, R = scene_with_box()
    m0 = points_in_box(pts, c[0], d[0], R[0])
    if which == "mirror":
        p2, c2, R2 = T.mirror_y(pts, c, R); d2 = d
        Tm = MIRROR_Y
    elif which == "rotz":
        th = 0.3; p2, c2, R2 = T.rotate_z(pts, c, R, th); d2 = d
        Tm = Rz(th)
    else:
        s = 1.04; p2, c2, d2 = T.scale(pts, c, d, s); R2 = R
        Tm = s * np.eye(3)
    # 1) exactly the same points are inside the box
    assert np.array_equal(m0, points_in_box(p2, c2[0], d2[0], R2[0]))
    # 2) the box is the same physical cuboid (transformed corners as a set)
    assert same_point_set(box_corners(c2[0], d2[0], R2[0]), box_corners(c[0], d[0], R[0]) @ Tm.T)
    # 3) still a proper rotation
    assert np.isclose(np.linalg.det(R2[0]), 1)
    # 4) inputs untouched
    assert not np.shares_memory(p2, pts)


def test_mirror_semantics():
    _, c, d, R = scene_with_box()
    _, _, R2 = T.mirror_y(np.zeros((1, 4)), c, R)
    R, R2 = R[0], R2[0]
    assert np.allclose(R2[:, 2], MIRROR_Y @ R[:, 2])   # head still points to mirrored head
    assert np.allclose(R2[:, 0], MIRROR_Y @ R[:, 0])   # chest normal mirrored
    assert np.allclose(R2[:, 1], -MIRROR_Y @ R[:, 1])  # left/right shoulders swap
    assert np.isclose(tilt_deg(R2), tilt_deg(R))       # mirror can't change tilt
    # the naive version M @ R is NOT a rotation -- this is the classic bug
    assert np.isclose(np.linalg.det(MIRROR_Y @ R), -1)


def test_mirror_is_involution():
    pts, c, d, R = scene_with_box()
    p2, c2, R2 = T.mirror_y(*T.mirror_y(pts, c, R))
    assert np.allclose(p2, pts) and np.allclose(c2, c) and np.allclose(R2, R)
    c3, R3 = T.mirror_boxes_only(*T.mirror_boxes_only(c, R))
    assert np.allclose(c3, c) and np.allclose(R3, R)


# ----------------------------------------------------------------- submission writer
@pytest.mark.parametrize("ext", [".json", ".jsonl.gz"])
def test_writer_roundtrip(ext):
    import gzip
    R = rand_R(3)
    preds = {"scene_0001/000000": dict(c=rng.normal(size=(3, 3)), d=np.ones((3, 3)),
                                       R=R, score=np.array([0.2, 0.9, 0.5])),
             "scene_0001/000001": dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)),
                                       R=np.zeros((0, 3, 3)), score=np.zeros(0))}
    path = os.path.join(tempfile.mkdtemp(), "sub" + ext)
    write_submission(preds, path, tokens=list(preds) + ["scene_0001/000002"])
    if ext == ".json":
        res = json.load(open(path))["results"]
    else:
        res = {j["sample_token"]: j["boxes"] for j in map(json.loads, gzip.open(path, "rt"))}
    assert set(res) == {"scene_0001/000000", "scene_0001/000001", "scene_0001/000002"}
    b = res["scene_0001/000000"]
    assert [x["score"] for x in b] == [0.9, 0.5, 0.2]          # sorted by score
    order = [1, 2, 0]
    for k, i in enumerate(order):
        assert rot_angle_deg(quat_to_R(b[k]["quaternion"]), R[i]) < 1e-3
    assert res["scene_0001/000002"] == []


Writing /kaggle/working/code/tests/test_geometry.py


In [23]:
%%writefile /kaggle/working/code/tests/test_post.py
"""Post-processing tests on synthetic tracks where the truth is known."""
import os, sys
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.geometry import quat_to_R, sym_angle_deg, Rx, Rz, SYM, tilt_deg
from d3.post import postprocess, fuse, fuse_frame, DEFAULTS
from d3.analyze import match

rng = np.random.default_rng(0)


def scene(n=200, flip_rate=0.1, noise=0.15, rot_noise=12, seed=0):
    rng = np.random.default_rng(seed)
    """One diver walking slowly; noisy detections; some head flips; FP blips in empty frames."""
    gt, pr = {}, {}
    R0 = Rz(0.7) @ Rx(0.2)
    for f in range(n):
        tok = f"scene_0001/{f:06d}"
        if 150 <= f < 170:                                   # empty stretch with a blip
            gt[tok] = dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)), R=np.zeros((0, 3, 3)), score=np.zeros(0))
            blip = f == 160
            pr[tok] = dict(c=np.array([[6.0, 2.0, 0.0]]) if blip else np.zeros((0, 3)),
                           d=np.ones((int(blip), 3)), R=np.tile(np.eye(3), (int(blip), 1, 1)),
                           score=np.array([0.5]) if blip else np.zeros(0))
            continue
        c = np.array([3 + 0.01 * f, -1 + 0.005 * f, 0.2])
        R = Rz(0.002 * f) @ R0
        gt[tok] = dict(c=c[None], d=np.array([[0.9, 0.82, 1.7]]), R=R[None], score=np.ones(1))
        ax = rng.normal(size=3); ax /= np.linalg.norm(ax)
        from scipy.spatial.transform import Rotation
        Rn = Rotation.from_rotvec(ax * np.radians(rng.normal(0, rot_noise))).as_matrix() @ R
        if rng.random() < flip_rate:
            Rn = Rn @ Rx(np.pi)
        if rng.random() < 0.5:
            Rn = Rn @ SYM                                     # free symmetric twin, must be harmless
        pr[tok] = dict(c=(c + rng.normal(0, noise, 3))[None], d=np.array([[0.9, 0.82, 1.7]]) * rng.uniform(0.85, 1.15),
                       R=Rn[None], score=np.array([rng.uniform(0.6, 0.9)]))
    return gt, pr


def errs(pr, gt):
    pairs, *_ = match(pr, gt, thr=0.0)
    pR = np.array([p["pR"] for p in pairs]); gR = np.array([p["gR"] for p in pairs])
    return (np.mean([np.linalg.norm(p["pc"] - p["gc"]) for p in pairs]), sym_angle_deg(pR, gR).mean(),
            np.mean(np.einsum("ni,ni->n", pR[:, :, 2], gR[:, :, 2]) < 0))


def test_defaults_are_identity():
    gt, pr = scene()
    out = postprocess(pr, {})
    for t in pr:
        assert np.allclose(out[t]["c"], pr[t]["c"]) and np.allclose(out[t]["R"], pr[t]["R"])


def test_smoothing_reduces_errors_and_flips():
    gt, pr = scene()
    c0, r0, f0 = errs(pr, gt)
    out = postprocess(pr, dict(sig_c=3.0, sig_r=2.0, flip_vote=1))
    c1, r1, f1 = errs(out, gt)
    print(f"center {c0:.3f}->{c1:.3f}  rot {r0:.1f}->{r1:.1f}  flips {f0:.3f}->{f1:.3f}")
    assert c1 < 0.6 * c0
    assert r1 < 0.5 * r0
    assert f1 < 0.35 * f0     # i.i.d. flips: a window can hold a local flipped majority


def test_rescore_suppresses_blip():
    gt, pr = scene()
    out = postprocess(pr, dict(min_len=3, short_pen=0.3))
    assert out["scene_0001/000160"]["score"][0] < 0.2
    assert out["scene_0001/000010"]["score"][0] > 0.5


def test_fuse_identical_models_and_symmetric_twins():
    gt, pr = scene()
    tw = {t: dict(c=p["c"], d=p["d"], R=p["R"] @ SYM if len(p["R"]) else p["R"], score=p["score"]) for t, p in pr.items()}
    f = fuse([pr, tw])
    for t in pr:
        assert len(f[t]["c"]) == len(pr[t]["c"])
        if len(pr[t]["c"]):
            assert np.allclose(f[t]["c"], pr[t]["c"])
            assert np.allclose(f[t]["score"], pr[t]["score"])
            assert sym_angle_deg(f[t]["R"][0], pr[t]["R"][0]) < 1e-3     # twin averaging is not garbage


def test_fuse_missing_model_halves_score():
    p = dict(c=np.array([[3.0, 0, 0]]), d=np.ones((1, 3)), R=np.eye(3)[None], score=np.array([0.8]))
    e = dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)), R=np.zeros((0, 3, 3)), score=np.zeros(0))
    f = fuse_frame([p, e], [1.0, 1.0], 0.6)
    assert np.isclose(f["score"][0], 0.4)


def test_flip_tilt():
    R = Rx(np.radians(170))[None]
    p = {"scene_0001/000000": dict(c=np.zeros((1, 3)), d=np.ones((1, 3)), R=R, score=np.ones(1))}
    out = postprocess(p, dict(flip_tilt=120))
    assert tilt_deg(out["scene_0001/000000"]["R"][0]) < 15


Writing /kaggle/working/code/tests/test_post.py


In [24]:
%%writefile /kaggle/working/code/tools/data_report.py
"""
Answers the planning questions with numbers instead of guesses (~3-5 min, CPU).

    python tools/data_report.py --root /kaggle/input/<...>/diver3d_v2

Q1 Pose mix per split, incl. head-down share       -> rebalancing, head-flip head?
Q2 Is the sonar static between frames?              -> is multi-frame input worth it?
Q3 Are frames contiguous, and how fast do divers move? -> tracking/Kalman settings
Q4 How stable are box dims within one track?        -> size smoothing / priors
Q5 How much GT does each crop lose?                 -> crop choice
"""
import argparse, os, sys, collections
import numpy as np
from scipy.spatial import cKDTree
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.io import load_gt, load_bin, frame_path, sort_tokens
from d3.geometry import tilt_deg, box_corners

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
args = ap.parse_args()
rng = np.random.default_rng(0)
G = {s: load_gt(os.path.join(args.root, "annotations", f"gt_{s}.json")) for s in ("train", "val")}

print("\n== Q1 pose mix (tilt = angle between body head-axis and sonar up) ==")
for s, gt in G.items():
    t = np.concatenate([tilt_deg(v["R"]) for v in gt.values() if len(v["c"])])
    print(f"{s:5s} n={len(t):5d} upright<15 {np.mean(t<15)*100:5.1f}%  15-75 {np.mean((t>=15)&(t<75))*100:5.1f}%"
          f"  horiz75-105 {np.mean((t>=75)&(t<=105))*100:5.1f}%  head-down>105 {np.mean(t>105)*100:5.1f}%")
    n = collections.Counter(len(v["c"]) for v in gt.values())
    print(f"      divers/frame: {dict(sorted(n.items()))}")

print("\n== Q2 is the sonar static? (fraction of frame-t points within 5 cm of frame t-1) ==")
tr = sort_tokens(G["train"].keys())
def overlap(a, b):
    A = load_bin(frame_path(args.root, a))[:, :3]; B = load_bin(frame_path(args.root, b))[:, :3]
    d, _ = cKDTree(B).query(A, k=1)
    return np.mean(d < 0.05), np.median(d)
same, diff = [], []
for _ in range(40):
    i = rng.integers(1, len(tr))
    if tr[i].split("/")[0] == tr[i - 1].split("/")[0]:
        same.append(overlap(tr[i], tr[i - 1]))
    j = rng.integers(len(tr))                       # control: a far-away frame
    if abs(j - i) > 100:
        diff.append(overlap(tr[i], tr[j]))
same, diff = np.array(same), np.array(diff)
print(f"consecutive frames : {same[:,0].mean()*100:5.1f}% within 5cm, median NN dist {np.median(same[:,1])*100:.1f} cm")
if len(diff): print(f"far-apart frames   : {diff[:,0].mean()*100:5.1f}% within 5cm, median NN dist {np.median(diff[:,1])*100:.1f} cm  (control)")
print("-> if consecutive >> control, background is stable and stacking frames can help.")

print("\n== Q3 frame contiguity and diver motion ==")
for s, gt in G.items():
    toks = sort_tokens(gt.keys()); gaps = collections.Counter()
    steps, rot_steps = [], []
    prev = {}
    for t in toks:
        sc, fr = t.split("/"); fr = int(fr)
        if sc in prev and fr - prev[sc][0] != 1: gaps[fr - prev[sc][0]] += 1
        v = gt[t]; cur = {int(i): (v["c"][k], v["R"][k]) for k, i in enumerate(v["iid"])}
        if sc in prev and fr - prev[sc][0] == 1:
            for i, (c, R) in cur.items():
                if i in prev[sc][1]:
                    steps.append(np.linalg.norm(c - prev[sc][1][i][0]))
                    tr_ = np.trace(prev[sc][1][i][1].T @ R)
                    rot_steps.append(np.degrees(np.arccos(np.clip((tr_ - 1) / 2, -1, 1))))
        prev[sc] = (fr, cur)
    print(f"{s:5s} frame gaps != 1: {dict(gaps) or 'none'}")
    print(f"      per-frame center step (m)  p50 {np.median(steps):.3f}  p90 {np.percentile(steps,90):.3f}  p99 {np.percentile(steps,99):.3f}")
    print(f"      per-frame rotation step (deg) p50 {np.median(rot_steps):.1f}  p90 {np.percentile(rot_steps,90):.1f}  p99 {np.percentile(rot_steps,99):.1f}")

print("\n== Q4 dims stability (std within a track vs across all boxes) ==")
for s, gt in G.items():
    tracks = collections.defaultdict(list)
    for t, v in gt.items():
        for k, i in enumerate(v["iid"]): tracks[(t.split('/')[0], int(i))].append(v["d"][k])
    allD = np.concatenate([np.array(x) for x in tracks.values()])
    within = np.mean([np.std(np.array(x), 0) for x in tracks.values() if len(x) > 20], 0)
    print(f"{s:5s} median dims {np.median(allD,0).round(2)}  global std {allD.std(0).round(3)}  within-track std {within.round(3)}")

print("\n== Q5 GT kept by each crop (center inside / all 8 corners inside) ==")
crops = {"paper [0,12]x[-5,5]x[-2.5,2.5]": ((0, 12), (-5, 5), (-2.5, 2.5)),
         "wider [0,14]x[-6,6]x[-3,3]":     ((0, 14), (-6, 6), (-3, 3))}
for s, gt in G.items():
    C = np.concatenate([v["c"] for v in gt.values() if len(v["c"])])
    K = np.concatenate([np.stack([box_corners(c, d, R) for c, d, R in zip(v["c"], v["d"], v["R"])])
                        for v in gt.values() if len(v["c"])])
    for name, (xr, yr, zr) in crops.items():
        inside = lambda P: ((P[..., 0] >= xr[0]) & (P[..., 0] < xr[1]) & (P[..., 1] >= yr[0]) &
                            (P[..., 1] < yr[1]) & (P[..., 2] >= zr[0]) & (P[..., 2] < zr[1]))
        print(f"{s:5s} {name:34s} centers {inside(C).mean()*100:6.2f}%   whole box {inside(K).all(1).mean()*100:6.2f}%")
    print(f"      center z range [{C[:,2].min():.2f}, {C[:,2].max():.2f}]   corner z range [{K[...,2].min():.2f}, {K[...,2].max():.2f}]")


Writing /kaggle/working/code/tools/data_report.py


In [25]:
%%writefile /kaggle/working/code/tools/final.py
"""
Final assembly, fully automatic:
  1. find every model's predictions under --search (prefers mirror-TTA files)
  2. score each model on val (official scorer)
  3. forward ensemble selection: start from the best model, add the model that raises
     fused val SDS the most; stop when no addition gains >= --min_gain
  4. tune post-processing on the chosen ensemble (tools/tune_post.py, both-halves rule)
  5. write + validate the test submission, and a summary.

  python tools/final.py --root ROOT --search /kaggle/input /kaggle/working/runs --out /kaggle/working/final
"""
import argparse, glob, json, os, subprocess, sys, time
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.infer import load_preds
from d3.io import write_submission
from d3.evaluate import run_scorer, fmt
from d3.post import fuse

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
ap.add_argument("--search", nargs="+", required=True)
ap.add_argument("--out", default="/kaggle/working/final")
ap.add_argument("--min_gain", type=float, default=0.002)
ap.add_argument("--max_models", type=int, default=6)
ap.add_argument("--quick", action="store_true")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)
gt_path = os.path.join(args.root, "annotations", "gt_val.json")
tokens = list(json.load(open(gt_path))["results"].keys())

# ---- 1. discover every (run, variant) that has both val and test predictions
variants = {}                                   # name -> {"plain": {...}, "tta": {...}}
for base in args.search:
    for v in glob.glob(os.path.join(base, "**", "raw_val*.pkl.gz"), recursive=True):
        d = os.path.dirname(v)
        name = os.path.basename(d)
        kind = "tta" if v.endswith("_tta.pkl.gz") else "plain"
        t = os.path.join(d, "raw_test_tta.pkl.gz" if kind == "tta" else "raw_test.pkl.gz")
        if os.path.exists(t) and kind not in variants.get(name, {}):
            variants.setdefault(name, {})[kind] = dict(val=v, test=t)
print("models found:")
for n, vs in sorted(variants.items()):
    print(f"  {n:4s} variants {sorted(vs)}  {next(iter(vs.values()))['val']}")
assert variants, "no models with both raw_val and raw_test found"

vals = {}


def score(keys):
    for k in keys:
        if k not in vals:
            n, kind = k
            vals[k] = load_preds(variants[n][kind]["val"])
    p = fuse([vals[k] for k in keys])
    path = write_submission(p, os.path.join(args.out, "tmp_sel.json"), tokens)
    return run_scorer(args.root, path, gt_path)


# ---- 2. single-model scores; keep the better variant (with / without mirror TTA) per model
single, models = {}, {}
for n in sorted(variants):
    res = {kind: score([(n, kind)]) for kind in variants[n]}
    for kind, m in res.items():
        print(f"single {n} {kind:5s}: {fmt(m)}", flush=True)
    kind = max(res, key=lambda k: res[k]["SDS"])
    models[n] = dict(**variants[n][kind], kind=kind)
    single[n] = res[kind]
    vals[n] = vals[(n, kind)]
print("chosen variants:", {n: m["kind"] for n, m in models.items()})


def score_names(names):
    p = fuse([vals[n] for n in names])
    path = write_submission(p, os.path.join(args.out, "tmp_sel.json"), tokens)
    return run_scorer(args.root, path, gt_path)

# ---- 3. forward selection
chosen = [max(single, key=lambda n: single[n]["SDS"])]
best = single[chosen[0]]
print(f"\nstart ensemble with {chosen} SDS {best['SDS']:.4f}")
while len(chosen) < args.max_models:
    cand = [n for n in models if n not in chosen]
    if not cand:
        break
    trials = {n: score_names(chosen + [n]) for n in cand}
    for n, m in trials.items():
        print(f"  + {n}: SDS {m['SDS']:.4f} ({m['SDS']-best['SDS']:+.4f})", flush=True)
    n_best = max(trials, key=lambda n: trials[n]["SDS"])
    if trials[n_best]["SDS"] - best["SDS"] < args.min_gain:
        break
    chosen.append(n_best); best = trials[n_best]
    print(f"ADD {n_best} -> {chosen} SDS {best['SDS']:.4f}", flush=True)
print(f"\nENSEMBLE {chosen}: {fmt(best)}")

# ---- 4+5. tune post-processing and write the submission
cmd = [sys.executable, os.path.join(os.path.dirname(os.path.abspath(__file__)), "tune_post.py"),
       "--root", args.root, "--out", args.out,
       "--val"] + [models[n]["val"] for n in chosen] + ["--test"] + [models[n]["test"] for n in chosen]
if args.quick:
    cmd.append("--quick")
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log = []
for line in p.stdout:
    print(line, end="", flush=True); log.append(line)
p.wait()

summary = dict(chosen=chosen, single={n: single[n]["SDS"] for n in single}, ensemble_raw=best,
               tuned=[l.strip() for l in log if l.startswith("VAL:")], time=time.ctime())
json.dump(summary, open(os.path.join(args.out, "SUMMARY.json"), "w"), indent=1)
print("\n==== SUMMARY ====")
print(json.dumps(summary, indent=1))
print("submission file:", os.path.join(args.out, "submission_test.json"))


Writing /kaggle/working/code/tools/final.py


In [26]:
%%writefile /kaggle/working/code/tools/tune_post.py
"""
Tune post-processing on val with the OFFICIAL scorer, then write the test submission.

  python tools/tune_post.py --root ROOT --val runs/B/raw_val.pkl.gz [runs/A/raw_val.pkl.gz ...] \
         --test runs/B/raw_test.pkl.gz [...] --out /kaggle/working/post

Overfitting guard: val is split by SCENE into two halves (4 + 3 scenes). A parameter
change is accepted only if it improves the full-val SDS by >= min_gain AND does not make
either half worse by more than 0.002. Changes that help one half and hurt the other are
noise or scene-specific -- exactly what would not transfer to test.
"""
import argparse, json, os, sys, time
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.infer import load_preds
from d3.io import load_gt, write_submission, sort_tokens
from d3.evaluate import run_scorer, fmt
from d3.post import postprocess, fuse, DEFAULTS
from d3.analyze import report

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
ap.add_argument("--val", nargs="+", required=True)
ap.add_argument("--test", nargs="*", default=[])
ap.add_argument("--weights", nargs="*", type=float, default=None)
ap.add_argument("--out", default="/kaggle/working/post")
ap.add_argument("--min_gain", type=float, default=0.001)
ap.add_argument("--quick", action="store_true", help="smaller search grid")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)

gt_path = os.path.join(args.root, "annotations", "gt_val.json")
raw = json.load(open(gt_path))["results"]
gt = load_gt(gt_path)
tokens = list(raw.keys())
scenes = sorted({t.split("/")[0] for t in tokens})
halves = [scenes[0::2], scenes[1::2]]
half_paths = []
for h, sc in enumerate(halves):
    p = os.path.join(args.out, f"gt_val_half{h}.json")
    json.dump({"split": "val", "results": {t: v for t, v in raw.items() if t.split("/")[0] in sc}}, open(p, "w"))
    half_paths.append(p)
print("val halves:", halves)

vals = [load_preds(p) for p in args.val]
base = fuse(vals, args.weights) if len(vals) > 1 or any("a" in next(iter(v.values())) for v in vals) else vals[0]
n_eval = [0]


def evaluate(P, tag="cur"):
    t0 = time.time()
    pp = postprocess(base, P)
    path = write_submission(pp, os.path.join(args.out, f"tmp_{tag}.json"), tokens)
    full = run_scorer(args.root, path, gt_path)
    hs = [run_scorer(args.root, path, hp)["SDS"] for hp in half_paths]
    n_eval[0] += 1
    return full, hs, time.time() - t0


print("\n== error analysis of the raw predictions ==")
report(base, gt)

P = dict(DEFAULTS)
best, best_h, _ = evaluate(P)
print(f"\nRAW: {fmt(best)} | halves {best_h[0]:.4f} {best_h[1]:.4f}")

GRID = [
    ("flip_tilt", [150.0, 120.0, 100.0]),
    ("sig_c", [1.0, 2.0, 3.0, 5.0]),
    ("sig_r", [1.0, 2.0, 4.0]),
    ("flip_vote", [1]),
    ("sig_d", [2.0, 5.0, 10.0]),
    ("a_score", [0.3, 0.5, 0.7]),
    ("min_len", [3, 5]),
    ("short_pen", [0.3, 0.6]),
    ("dim_scale", [(0.95, 0.95, 0.95), (0.9, 0.9, 0.9), (1.0, 1.0, 0.95), (0.95, 0.95, 1.0)]),
    ("track_thr", [0.1, 0.2]),
    ("gate", [0.6, 1.0]),
    ("final_thr", [0.05, 0.1]),
    ("max_det", [4, 6]),
]
if args.quick:
    GRID = [(k, v[:2]) for k, v in GRID]

for rnd in range(2):                                  # two passes of coordinate descent
    for key, values in GRID:
        if key == "short_pen" and P["min_len"] <= 1:
            continue
        for v in values:
            if P[key] == v:
                continue
            trial = {**P, key: v}
            m, h, sec = evaluate(trial)
            gain = m["SDS"] - best["SDS"]
            ok = gain >= args.min_gain and all(hn >= ho - 0.002 for hn, ho in zip(h, best_h))
            print(f"[{rnd}] {key}={v}: SDS {m['SDS']:.4f} ({gain:+.4f}) halves {h[0]:.4f} {h[1]:.4f} "
                  f"{'ACCEPT' if ok else ''} ({sec:.0f}s)", flush=True)
            if ok:
                P, best, best_h = trial, m, h

print("\n== BEST post-processing ==")
print(json.dumps({k: P[k] for k in P if P[k] != DEFAULTS[k]}, indent=1))
print("VAL:", fmt(best), "| halves", [round(x, 4) for x in best_h], f"| {n_eval[0]} scorer runs")
json.dump(P, open(os.path.join(args.out, "best_post.json"), "w"), indent=1)
print("\n== error analysis AFTER post-processing ==")
report(postprocess(base, P), gt)

if args.test:
    tests = [load_preds(p) for p in args.test]
    tb = fuse(tests, args.weights) if len(tests) > 1 or any("a" in next(iter(v.values())) for v in tests) else tests[0]
    test_tokens = [l.strip() for l in open(os.path.join(args.root, "annotations", "test_tokens.txt")) if l.strip()]
    sub = write_submission(postprocess(tb, P), os.path.join(args.out, "submission_test.json"), test_tokens)
    import subprocess
    r = subprocess.run([sys.executable, os.path.join(args.root, "tools", "validate_submission.py"),
                        "--pred", sub, "--tokens", os.path.join(args.root, "annotations", "test_tokens.txt")],
                       capture_output=True, text=True)
    print("\nwrote", sub, f"({os.path.getsize(sub)/1e6:.1f} MB)")
    print("validate_submission:", (r.stdout + r.stderr)[-600:])


Writing /kaggle/working/code/tools/tune_post.py


## Settings

In [27]:
GPU_WORK = True          # False = CPU-only fallback: just ensemble + tune the attached predictions
EPOCHS = 20
NEW_RUNS = {"F": "rebalance=1.0 seed=4",      # GPU 0: replicate of the best config (C)
            "G": "rebalance=1.25 seed=5"}     # GPU 1: a bit more pose rebalancing

CODE = "/kaggle/working/code"
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"FAILED: {cmd}"
def stage(name, fn):
    """Run a stage; on failure print the error and CONTINUE (so later stages still produce a submission)."""
    t0 = time.time(); print(f"\n######## {name} ########", flush=True)
    try:
        fn(); print(f"######## {name} OK ({(time.time()-t0)/60:.1f} min)", flush=True)
    except Exception:
        traceback.print_exc(); print(f"######## {name} FAILED -- continuing", flush=True)
print("attached checkpoints:", glob.glob("/kaggle/input/**/runs/*/best.pt", recursive=True))
print("attached predictions:", glob.glob("/kaggle/input/**/runs/*/raw_val.pkl.gz", recursive=True))
sh("pip -q install pytest && python -m pytest -q tests/test_geometry.py tests/test_post.py")

attached checkpoints: ['/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/B/best.pt', '/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/C/best.pt']
attached predictions: ['/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/B/raw_val.pkl.gz', '/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/C/raw_val.pkl.gz']
.........................                                                [100%]
25 passed in 17.04s


## 1. Train F and G (one per GPU)

In [28]:
def train_new():
    os.makedirs("/kaggle/working/runs", exist_ok=True)
    sh(f"""python -c "
import sys; sys.path.insert(0,'.')
from d3.io import load_gt; from d3.data import GTDatabase
GTDatabase('{ROOT}', load_gt('{ROOT}/annotations/gt_train.json'), '/kaggle/working/runs/gtdb.npz')" """)
    procs = {}
    for gpu, (name, sets) in enumerate(NEW_RUNS.items()):
        out = f"/kaggle/working/runs/{name}"
        cmd = f"python -m d3.train --root {ROOT} --out {out} --gpu {gpu} --set epochs={EPOCHS} name={name} {sets}"
        print(cmd, flush=True)
        procs[name] = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=open(out + ".stdout", "w"), stderr=subprocess.STDOUT)
        time.sleep(20)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(300)
        for name in procs:
            try:
                lines = open(f"/kaggle/working/runs/{name}/log.txt").read().strip().splitlines()
                print(time.strftime("%H:%M"), name, "|", " || ".join(l for l in lines if "==" in l or "**" in l)[-300:], flush=True)
            except FileNotFoundError:
                pass
    for name, p in procs.items():
        print(name, "exit code", p.returncode, open(f"/kaggle/working/runs/{name}.stdout").read()[-600:])
if GPU_WORK:
    stage("TRAIN F,G", train_new)


######## TRAIN F,G ########
python -m d3.train --root /tmp/data/diver3d_v2 --out /kaggle/working/runs/F --gpu 0 --set epochs=20 name=F rebalance=1.0 seed=4
python -m d3.train --root /tmp/data/diver3d_v2 --out /kaggle/working/runs/G --gpu 1 --set epochs=20 name=G rebalance=1.25 seed=5
20:48 F | 
20:48 G | 
20:53 F | == epoch 0 done in 8.4 min | hm 0.627 off 0.416 z 0.146 dim 0.474 rot 1.255 rot_deg 40.416
20:53 G | == epoch 0 done in 8.2 min | hm 0.587 off 0.408 z 0.143 dim 0.472 rot 1.185 rot_deg 38.883
20:58 F | == epoch 0 done in 8.4 min | hm 0.627 off 0.416 z 0.146 dim 0.474 rot 1.255 rot_deg 40.416
20:58 G | == epoch 0 done in 8.2 min | hm 0.587 off 0.408 z 0.143 dim 0.472 rot 1.185 rot_deg 38.883
21:03 F | == epoch 0 done in 8.4 min | hm 0.627 off 0.416 z 0.146 dim 0.474 rot 1.255 rot_deg 40.416 || == epoch 1 done in 7.9 min | hm 0.482 off 0.386 z 0.123 dim 0.428 rot 0.991 rot_deg 34.220
21:03 G | == epoch 0 done in 8.2 min | hm 0.587 off 0.408 z 0.143 dim 0.472 rot 1.185 rot_deg

## 2. Mirror-TTA predictions for every model

In [29]:
def predict_all():
    ckpts = {}
    for p in sorted(glob.glob("/kaggle/input/**/runs/*/best.pt", recursive=True)) + \
             sorted(glob.glob("/kaggle/working/runs/*/best.pt")):
        name = os.path.basename(os.path.dirname(p))
        if name in ("overfit", "speed"):
            continue
        ckpts[name] = p                     # working/ (new) overrides input/ for the same name
    print("models to predict:", ckpts, flush=True)
    queues = [[], []]
    for i, (name, p) in enumerate(sorted(ckpts.items())):
        out = os.path.dirname(p) if p.startswith("/kaggle/working") else f"/kaggle/working/tta/{name}"
        queues[i % 2].append(f"python -m d3.predict --root {ROOT} --ckpt {p} --out {out} --gpu {i % 2} --mirror")
    procs = [subprocess.Popen(" ; ".join(q), shell=True, cwd=CODE, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True) for q in queues if q]
    for pr in procs:
        for line in pr.stdout:
            if line.startswith(("VAL", "validate", "Traceback", "RuntimeError", "Error")):
                print(line, end="", flush=True)
        pr.wait()
if GPU_WORK:
    stage("MIRROR-TTA PREDICT", predict_all)


######## MIRROR-TTA PREDICT ########
models to predict: {'B': '/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/B/best.pt', 'C': '/kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/C/best.pt', 'F': '/kaggle/working/runs/F/best.pt', 'G': '/kaggle/working/runs/G/best.pt'}
VAL /kaggle/working/tta/B SDS 0.6406  mAP3D 0.6149  AP3D@0.50 0.1668  APBEV@0.35 0.7653  ATE_m 0.1984  ASE 0.3410  AOE3D_deg 41.5830  TILT_deg 17.3245
validate_submission: parsed 8112 frames / 29895 detections (4.9 MB on disk)
VALID (with warnings)
VAL /kaggle/working/runs/F SDS 0.6655  mAP3D 0.6579  AP3D@0.50 0.1601  APBEV@0.35 0.8222  ATE_m 0.2007  ASE 0.3411  AOE3D_deg 39.4878  TILT_deg 17.5865
validate_submission: parsed 8112 frames / 34085 detections (5.5 MB on disk)
VALID (with warnings)
VAL /kaggle/working/tta/C SDS 0.6608  mAP3D 0.6499  AP3D@0.50 0.1467  APBEV@0.35 0.8146  ATE_m 0.2026  ASE 0.3472  AOE3D_deg 39.1715  TILT_deg 17.5927
validate_submission: parsed 8112 frames / 30775 detections

## 3. Ensemble selection

In [30]:
stage("FINAL ASSEMBLY", lambda: sh(f"python tools/final.py --root {ROOT} --search /kaggle/input /kaggle/working --out /kaggle/working/final"))
for p in ["/kaggle/working/final/SUMMARY.json", "/kaggle/working/final/submission_test.json"]:
    print(p, "exists" if os.path.exists(p) else "MISSING", f"{os.path.getsize(p)/1e6:.1f} MB" if os.path.exists(p) else "")


######## FINAL ASSEMBLY ########
models found:
  B    variants ['plain', 'tta']  /kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/B/raw_val.pkl.gz
  C    variants ['plain', 'tta']  /kaggle/input/notebooks/akshitajawahar10/diver3d-model/runs/C/raw_val.pkl.gz
  F    variants ['plain', 'tta']  /kaggle/working/runs/F/raw_val.pkl.gz
  G    variants ['plain', 'tta']  /kaggle/working/runs/G/raw_val.pkl.gz
single B plain: SDS 0.6406  mAP3D 0.6149  AP3D@0.50 0.1668  APBEV@0.35 0.7653  ATE_m 0.1984  ASE 0.3410  AOE3D_deg 41.5830  TILT_deg 17.3245
single B tta  : SDS 0.6576  mAP3D 0.6453  AP3D@0.50 0.1847  APBEV@0.35 0.7861  ATE_m 0.1972  ASE 0.3359  AOE3D_deg 41.1603  TILT_deg 17.1215
single C plain: SDS 0.6608  mAP3D 0.6499  AP3D@0.50 0.1467  APBEV@0.35 0.8146  ATE_m 0.2026  ASE 0.3472  AOE3D_deg 39.1715  TILT_deg 17.5927
single C tta  : SDS 0.6637  mAP3D 0.6586  AP3D@0.50 0.1503  APBEV@0.35 0.8196  ATE_m 0.2014  ASE 0.3474  AOE3D_deg 40.0316  TILT_deg 17.2218
single F plain: SDS 0.